## K-1. Menyiapkan Lingkungan dan Data Contoh

Pustaka duckdb, networkx, shapely, tinydb, dan deltalake versi 1.0 ke atas dipasang melalui perintah *shell* pip install berargumen -q, lalu drive diimpor dari google.colab dan Google Drive disambungkan dengan drive.mount("/content/drive"). Konstanta DIR_P3 dan DIR_SIMPAN ditetapkan pada folder BigData/Praktikum3 dan BigData/Praktikum4 di Drive, sedangkan TRIPS_BERSIH disusun dari f-string atas DIR_P3. Direktori penyimpanan dibuat dengan os.makedirs() berargumen exist_ok=True, lalu assert os.path.exists(TRIPS_BERSIH) menghentikan sel beserta pesan rujukan ke P3/K-10 apabila folder hasil Praktikum 3 tidak ditemukan.

Modul os, time, json, sqlite3, random, shutil, dan glob diimpor bersama numpy beralias np dan pandas beralias pd. List kosong catatan disiapkan, lalu didefinisikan fungsi ukur(label, fungsi) yang mencatat time.perf_counter() sebelum memanggil fungsi(), menghitung selisihnya, menambahkan dict berisi langkah dan detik yang dibulatkan round(detik, 4) ke catatan, mencetak label beserta durasinya melalui f-string berformat .4f, dan me-*return* hasil fungsi. Terakhir, folder trips_bersih dibaca ke df melalui pd.read_parquet(TRIPS_BERSIH) di dalam ukur() sebagaimana tertulis pada modul, dibungkus blok *try-except* agar *exception* yang muncul dicetak bersama nama kelasnya dan Run all tidak terhenti.

Dugaan: pembacaan folder Parquet berpartisi diperkirakan berhasil dan selesai dalam hitungan detik, karena folder hasil Praktikum 3 sudah tersimpan dalam format kolumnar yang siap dibaca pandas tanpa penyesuaian tambahan.

In [ ]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

from google.colab import drive
drive.mount("/content/drive")

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"

import os
os.makedirs(DIR_SIMPAN, exist_ok=True)
assert os.path.exists(TRIPS_BERSIH), "Folder trips_bersih/ tidak ada. Jalankan ulang P3/K-10 atau pakai K-1b."


import os, time, json, sqlite3, random, shutil, glob
import numpy as np, pandas as pd

catatan = []
def ukur(label, fungsi):
  t0 = time.perf_counter()
  hasil = fungsi()
  detik = time.perf_counter() - t0
  catatan.append({"langkah": label, "detik": round(detik, 4)})
  print(f"[{label}] {detik:.4f} s")
  return hasil

try:
  df = ukur("baca trips_bersih", lambda: pd.read_parquet(TRIPS_BERSIH))
except Exception as e:
  print("GAGAL membaca trips_bersih/")
  print(type(e).__name__, ":", e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GAGAL membaca trips_bersih/
ArrowInvalid : Cannot yet unify dictionaries with nulls


## K-1. Catatan Error

Pembacaan folder trips_bersih berhenti dengan ArrowInvalid berbunyi Cannot yet unify dictionaries with nulls. Penyebabnya, penulisan berpartisi pada Praktikum 3 menghasilkan direktori borough_naik=__HIVE_DEFAULT_PARTITION__ untuk 340 perjalanan yang wilayah penjemputannya kosong. PyArrow membaca kolom partisi sebagai *dictionary*, lalu gagal menyatukan kamus antarpartisi ketika salah satu nilainya kosong. Kendala yang sama tercatat pada laporan Praktikum 3 subbab 3.6.4.

Solusinya, folder disalin ke disk runtime lalu berkas pada partisi kosong dipindahkan ke partisi borough_naik=Unknown yang memang sudah ada pada data, sehingga tidak ada baris yang dibuang dan tidak ada nilai kosong yang tersisa pada kolom partisi. Penggantian dilakukan pada salinan, sehingga hasil Praktikum 3 tidak berubah, dan seluruh baris modul berikutnya dapat dijalankan apa adanya.

## K-1. Solusi Terhadap Error

Konstanta TRIPS_RAPI ditetapkan pada /content/trips_bersih_rapi, lalu salinan folder dibuat dengan shutil.copytree() apabila os.path.exists() bernilai False. Apabila direktori borough_naik=__HIVE_DEFAULT_PARTITION__ ada pada salinan itu, direktori borough_naik=Unknown disiapkan melalui os.makedirs() berargumen exist_ok=True, seluruh berkas di dalam partisi kosong dipindahkan ke sana melalui *for loop* atas sorted(os.listdir()) yang memanggil shutil.move() dengan awalan nama kosong beserta nomor urutnya agar tidak bertabrakan dengan berkas yang sudah ada, lalu direktori yang telah kosong dihapus dengan os.rmdir() dan nama TRIPS_BERSIH diarahkan ke salinan tersebut. Pemisahan kedua *if statement* membuat sel ini aman dijalankan berulang, termasuk ketika penyalinan sudah terjadi tetapi penggabungan belum.

Setelah itu seluruh baris modul dijalankan apa adanya, yaitu folder dibaca ke df melalui pd.read_parquet(TRIPS_BERSIH) di dalam ukur(), dict NAMA_BARU memetakan delapan nama kolom asli TLC ke nama yang mudah dibaca dan dipakai oleh df.rename(columns=NAMA_BARU), kolom wilayah_jemput di-*cast* dengan astype(str), jumlah baris berpemisah ribuan beserta df.shape[1] dicetak, lalu contoh kerja dibentuk dengan df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True), diberi kolom id_perjalanan dari sample.index + 1, dan kolom waktu_jemput di-*cast* astype(str).

Keputusan pra-pemrosesan pada sel ini ada tiga. Pertama, perjalanan tanpa wilayah digabungkan ke kelompok Unknown, bukan dibuang, sehingga jumlah baris tetap utuh dan kelompok Unknown bertambah sekitar 340 perjalanan. Kedua, contoh dibatasi 300.000 baris dengan random_state=42 mengikuti modul agar K-2 dan K-3 tidak terlalu lama dan hasilnya dapat diulang, dengan konsekuensi angka K-2 dan K-3 tidak sebanding langsung dengan K-4 sampai K-6 yang memakai seluruh isi folder. Ketiga, waktu_jemput di-*cast* menjadi *string* karena json.dumps() pada K-2 dan TinyDB pada K-3 tidak dapat menyimpan objek Timestamp, dengan konsekuensi kolom itu tidak lagi dapat dihitung sebagai waktu pada kedua langkah tersebut.

Dugaan: pembacaan salinan yang sudah rapi diperkirakan selesai dalam hitungan detik karena berkas Parquet dibaca dari disk runtime, bukan dari Google Drive, dan jumlah kelompok wilayah diperkirakan tidak bertambah karena partisi Unknown memang sudah ada sebelumnya.

In [ ]:
# partisi bernilai kosong dari Praktikum 3 digabungkan ke partisi Unknown pada salinan di runtime
TRIPS_RAPI = "/content/trips_bersih_rapi"
PARTISI_KOSONG = f"{TRIPS_RAPI}/borough_naik=__HIVE_DEFAULT_PARTITION__"
PARTISI_UNKNOWN = f"{TRIPS_RAPI}/borough_naik=Unknown"

if not os.path.exists(TRIPS_RAPI):
  shutil.copytree(TRIPS_BERSIH, TRIPS_RAPI)
if os.path.exists(PARTISI_KOSONG):
  os.makedirs(PARTISI_UNKNOWN, exist_ok=True)
  for i, nama in enumerate(sorted(os.listdir(PARTISI_KOSONG))):
    shutil.move(f"{PARTISI_KOSONG}/{nama}", f"{PARTISI_UNKNOWN}/kosong-{i}-{nama}")
  os.rmdir(PARTISI_KOSONG)
TRIPS_BERSIH = TRIPS_RAPI

df = ukur("baca trips_bersih", lambda: pd.read_parquet(TRIPS_BERSIH))

NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}

df = df.rename(columns = NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris: ", f"{len(df):,}", "| kolom:", df.shape[1])

sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index+1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

[baca trips_bersih] 1.9129 s
baris:  2,986,910 | kolom: 18


## K-2. Tabel Biasa vs Key-value

Delapan nama kolom ditetapkan pada list KOLOM, lalu data dibentuk dari sample[KOLOM].copy(). Model pertama berupa tabel SQLite, yaitu berkas uji.db dihapus dengan os.remove() apabila os.path.exists() bernilai True, koneksi dibuka dengan sqlite3.connect("uji.db"), dan tabel perjalanan dimuat melalui data.to_sql("perjalanan", con, index=False) di dalam ukur(). Model kedua berupa key-value, yaitu fungsi bangun_kv() me-*return* *dict comprehension* atas data.to_dict("records") yang kuncinya berpola perjalanan:<id> dan nilainya teks JSON dari json.dumps(), dijalankan di dalam ukur() lalu jumlah kuncinya dicetak dengan len(kv).

Pertanyaan pertama menguji pengambilan satu data, yaitu random.seed(42) menetapkan *seed* dan id_uji diisi random.sample(range(1, len(data) + 1), 200). Fungsi cari_sql() membuka kursor dengan con.cursor() lalu me-*return* list hasil cur.execute() berparameter tanda tanya yang diikuti fetchone(), sedangkan cari_kv() me-*return* list json.loads() atas kv. Keduanya diukur tiga kali, yaitu sebelum indeks, setelah con.execute() membuat indeks idx_id melalui CREATE INDEX, dan pada key-value, lalu kesamaan ketiga hasil diperiksa dengan rantai operator ==. Pertanyaan kedua menguji perhitungan atas semua data, yaitu hitung_sql() menjalankan SELECT dengan COUNT(\*) dan ROUND(AVG(total_bayar), 2) yang dikelompokkan GROUP BY wilayah_jemput dan diurutkan ORDER BY 2 DESC, 1, sedangkan hitung_kv() menjalankan *for loop* atas kv.values() yang membuka tiap nilai dengan json.loads(), menjumlahkan total_bayar dan cacah baris ke dalam dict total dan jumlah, lalu me-*return* sorted() dengan key berupa *lambda* bernilai negatif cacah dan nama wilayah. Tiga baris pertama hasil SQL dan kesamaan h1 dengan h2 dicetak.

Dugaan: pencarian pada tabel tanpa indeks diperkirakan paling lambat dengan selisih ratusan kali terhadap tabel berindeks, dan key-value diperkirakan tetap menang telak atas tabel berindeks dengan selisih sekitar sepuluh kali karena kamus Python tidak menyentuh berkas sama sekali, sedangkan pada pertanyaan kedua key-value diperkirakan kalah sekitar lima sampai sepuluh kali.

In [ ]:
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

if os.path.exists("uji.db"):
  os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

def bangun_kv():
  return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}

kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key: ", len(kv))

random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
  cur = con.cursor()
  return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]

def cari_kv():
  return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)")
r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan - key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

def hitung_sql():
  return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                    FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()


def hitung_kv():
  total, jumlah = {}, {}
  for teks in kv.values():
    d = json.loads(teks)
    w = d["wilayah_jemput"]
    total[w] = total.get(w, 0) + d["total_bayar"]
    jumlah[w] = jumlah.get(w, 0) + 1
  return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[relasional: muat tabel] 3.4134 s
[key-value: bangun dict] 10.0131 s
jumlah key:  300000
[cari 200 perjalanan - SQL tanpa indeks] 8.1600 s
[cari 200 perjalanan - SQL dengan indeks] 0.0040 s
[cari 200 perjalanan - key-value] 0.0013 s
hasil sama? True
[hitung per wilayah - SQL GROUP BY] 0.2040 s
[hitung per wilayah - key-value (buka semua nilai)] 1.4833 s
[('Manhattan', 267034, 22.7), ('Queens', 27288, 70.34), ('Unknown', 3824, 39.42)]
hasil sama? True


## K-3. Document: Bentuk Data yang Fleksibel

Kelas TinyDB dan Query diimpor dari tinydb, sedangkan MemoryStorage diimpor dari tinydb.storages. Didefinisikan fungsi ke_dokumen(r) yang menyusun dict bersarang berisi id_perjalanan hasil int(), waktu_jemput apa adanya, serta tiga objek anak, yaitu jemput berisi zona hasil int(r.zona_jemput) dan wilayah, tujuan berisi zona hasil int(r.zona_tujuan), dan biaya berisi jarak_mil, total, serta metode_bayar dengan nilai angka di-*cast* float(). Pada *if statement* di dalamnya, syarat r.metode_bayar sama dengan Kartu Kredit, yaitu nilai yang benar-benar dipakai kolom nama_pembayaran pada trips_bersih, menambahkan kunci tip pada dok["biaya"] dari float(r.tip), sehingga field itu hanya ada di sebagian dokumen, lalu dok di-*return*.

Basis data dibentuk dengan TinyDB(storage=MemoryStorage) agar berjalan di memori, tabel perjalanan diambil melalui db.table(), dan docs disusun dengan *list comprehension* atas sample.head(20_000).itertuples() yang memanggil ke_dokumen(). Seluruh dokumen dimasukkan sekaligus melalui tabel.insert_multiple(docs), jumlahnya dicetak dengan len(tabel), dan satu contoh dokumen bersarang ditampilkan melalui json.dumps(tabel.all()[0], indent=2).

Keputusan pra-pemrosesan pada sel ini, dokumen dibatasi 20.000 baris pertama dari sample mengikuti modul agar penyusunan dokumen tidak memakan waktu lama, dengan konsekuensi populasi dokumen berbeda dengan tabel pada K-2 sehingga keduanya tidak dapat dibandingkan langsung, sebagaimana diuji pada Latihan 2.

Dugaan: hampir sembilan dari sepuluh dokumen diperkirakan memiliki field tip karena pembayaran kartu kredit mendominasi perjalanan taksi, sehingga dokumen tanpa field itu diperkirakan tinggal sedikit.

In [ ]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
  dok = {
      "id_perjalanan": int(r.id_perjalanan),
      "waktu_jemput": r.waktu_jemput,
      "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
      "tujuan": {"zona": int(r.zona_tujuan)},
      "biaya": {"jarak_mil": float(r.jarak_mil),
                "total": float(r.total_bayar),
                "metode_bayar": r.metode_bayar},
  }

  if r.metode_bayar == "Kartu Kredit":
    dok["biaya"]["tip"] = float(r.tip)
  return dok

db = TinyDB(storage = MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)
print("jumlah dokumen: ", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

jumlah dokumen:  20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-16 10:40:39",
  "jemput": {
    "zona": 48,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 164
  },
  "biaya": {
    "jarak_mil": 1.55,
    "total": 19.32,
    "metode_bayar": "Kartu Kredit",
    "tip": 3.22
  }
}


## K-3. Pencarian Dokumen dan Penambahan Field Baru

Objek Q dibentuk dari Query() sebagai alat penyusun syarat. Tiga pencarian dijalankan melalui tabel.search() dan jumlahnya dicetak dengan len(), yaitu gabungan Q.jemput.wilayah sama dengan Manhattan dan Q.biaya.total lebih dari 50 yang disambung operator &, lalu Q.biaya.tip.exists() untuk dokumen yang memiliki field tip, dan bentuk negasinya dengan operator ~ untuk dokumen yang tidak memilikinya. Setelah itu satu dokumen baru dimasukkan melalui tabel.insert() dengan id_perjalanan 999_999 yang membawa objek cuaca berisi hujan bernilai True dan suhu_c bernilai -2.0, yaitu field yang tidak dimiliki 20.000 dokumen sebelumnya. Jumlah dokumen yang memiliki field cuaca dicetak dari tabel.search(Q.cuaca.exists()) dan jumlah seluruh dokumen dari len(tabel).

Dugaan: pencarian dokumen yang dijemput di Manhattan dengan total di atas 50 diperkirakan menghasilkan puluhan dokumen karena Manhattan paling ramai, sedangkan penyisipan dokumen berfield cuaca diperkirakan diterima tanpa galat dan tanpa mengubah dokumen lama.

In [ ]:
Q = Query()
print("jemput di Manhattan & total > 50 :", len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50))))
print("dokumen yang punya field tip     :", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip     :", len(tabel.search(~Q.biaya.tip.exists())))

tabel.insert({"id_perjalanan": 999_999, "waktu_jemput": "2023-01-15 08:00:00",
              "jemput": {"zona": 161, "wilayah": "Manhattan"}, "tujuan": {"zona": 236},
              "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu Kredit", "tip": 3.0},
              "cuaca": {"hujan": True, "suhu_c": -2.0}})
print("dokumen yang punya field cuaca   :", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang           :", len(tabel))

jemput di Manhattan & total > 50 : 693
dokumen yang punya field tip     : 15906
dokumen yang TIDAK punya tip     : 4094
dokumen yang punya field cuaca   : 1
total dokumen sekarang           : 20001


## K-4. Penyimpanan per Kolom: DuckDB pada Parquet

Modul duckdb diimpor, pola berkas GLOB disusun dari f-string atas TRIPS_BERSIH dengan dua tingkat tanda bintang agar mencakup folder partisi, dan koneksi dibuka dengan duckdb.connect(). Melalui con_d.sql() dijalankan CREATE VIEW perjalanan yang memetakan tujuh kolom asli TLC ke nama baru dengan kata kunci AS dan membacanya dari read_parquet() berargumen hive_partitioning=true, sehingga view hanya menjadi jendela tanpa menyalin data. Query SQL berisi SELECT wilayah_jemput, COUNT(\*), dan ROUND(AVG(total_bayar), 2) yang dikelompokkan GROUP BY wilayah_jemput serta diurutkan ORDER BY jumlah DESC lalu wilayah_jemput, dijalankan di dalam ukur() dan diubah ke DataFrame melalui df(), lalu hasilnya dicetak.

Sebagai pembanding disiapkan SQLite yang menyimpan per baris, yaitu KOLOM_S berisi lima kolom, berkas baris.db dihapus dengan os.remove() apabila os.path.exists() bernilai True, koneksi dibuka dengan sqlite3.connect(), dan df[KOLOM_S] dimuat melalui to_sql("perjalanan", con_s, index=False) di dalam ukur(). Perhitungan yang sama dijalankan dengan con_s.execute() yang diakhiri fetchall(), lalu kesamaan kedua hasil diperiksa dengan membandingkan list tuple dari hasil_d.itertuples(index=False) terhadap list tuple dari hasil_s.

Dugaan: DuckDB diperkirakan belasan kali lebih cepat daripada SQLite pada perhitungan yang sama karena hanya dua kolom yang dibaca, dan selisihnya diperkirakan membesar lagi apabila waktu memuat tabel SQLite ikut dihitung karena DuckDB tidak melalui tahap muat sama sekali.

In [ ]:
import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

con_d.sql(f"""
          CREATE VIEW perjalanan AS
          SELECT tpep_pickup_datetime AS waktu_jemput,
          PULocationID AS zona_jemput,
          DOLocationID AS zona_tujuan,
          borough_naik AS wilayah_jemput,
          trip_distance AS jarak_mil,
          total_amount AS total_bayar,
          nama_pembayaran AS metode_bayar
          FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
        FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB - hitung per wilayah (langsung dari Parquet)", lambda:
               con_d.sql(SQL).df())
print(hasil_d)

KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
  os.remove("baris.db")
con_s = sqlite3.connect("baris.db")
ukur("SQLite - muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False))
hasil_s = ukur("SQLite - hitung per wilayah", lambda: con_s.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
              FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())
print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in hasil_s])

[DuckDB - hitung per wilayah (langsung dari Parquet)] 0.0965 s
  wilayah_jemput   jumlah  rata_bayar
0      Manhattan  2659609       22.72
1         Queens   270315       70.46
2        Unknown    37949       39.56
3       Brooklyn    15724       32.72
4          Bronx     3074       34.44
5  Staten Island      211       74.07
6            EWR       28      106.41
[SQLite - muat seluruh baris ke tabel] 9.2538 s
[SQLite - hitung per wilayah] 2.6813 s
hasil sama? True


## K-4. Filter pada Kolom Partisi dan Perbandingan Ukuran

Query SQL2 berisi SELECT COUNT(\*) dan ROUND(AVG(total_bayar), 2) yang disaring pada wilayah_jemput bernilai Staten Island, dijalankan melalui con_d.sql().fetchall() di dalam ukur() lalu dicetak, sehingga waktunya dapat dibandingkan dengan perhitungan seluruh wilayah pada sel sebelumnya. Ukuran folder Parquet dihitung dengan *generator expression* atas os.walk(TRIPS_BERSIH) yang menjumlahkan os.path.getsize(os.path.join(r, f)) untuk setiap berkas melalui sum(), lalu dibagi 1024 pangkat dua agar bersatuan MB dan di-*assign* ke ukuran_pq. Baris terakhir mencetak perbandingan ukuran melalui f-string berformat .1f, yaitu Parquet beserta df.shape[1] kolomnya dan berkas baris.db beserta len(KOLOM_S) kolomnya yang dibaca dengan os.path.getsize().

Dugaan: penyaringan satu wilayah diperkirakan beberapa kali lebih cepat daripada perhitungan seluruh wilayah karena hanya folder partisi wilayah itu yang dibuka, sedangkan berkas Parquet yang memuat lima belas kolom diperkirakan lebih besar daripada berkas SQLite yang hanya memuat lima kolom karena jumlah kolomnya tiga kali lipat.

In [ ]:
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
          FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB - hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))
ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(TRIPS_BERSIH) for f in fs) / 1024 ** 2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db') / 1024 **2:.1f} MB")

[DuckDB - hanya wilayah Staten Island] 0.0187 s
[(211, 74.07)]
Parquet (18 kolom): 66.5 MB | SQLite (5 kolom): 114.2 MB


## K-5. Graph: Pertanyaan tentang Keterhubungan

Pustaka networkx diimpor dengan alias nx. Tabel arus dibentuk melalui con_d.sql() yang menjalankan SELECT zona_jemput AS asal, zona_tujuan AS tujuan, dan COUNT(\*) AS jumlah dengan GROUP BY 1, 2, diubah ke DataFrame dengan df(), dan dibungkus ukur(). Graph berarah dibentuk dengan nx.DiGraph(), lalu *for loop* atas arus.itertuples(index=False) memanggil G.add_edge() dengan asal dan tujuan yang di-*cast* int() serta atribut jumlah sebagai bobot, dan jumlah node serta edge dicetak melalui G.number_of_nodes() dan G.number_of_edges().

Lima arus terbesar diambil dua kali untuk disilangkan. Pada sisi graph, sorted() atas G.edges(data=True) memakai key berupa *lambda* bernilai negatif bobot jumlah lalu simpul asal dan tujuan sebagai pengurut tambahan, dipotong lima teratas dengan *slicing*, dan dicetak sebagai list tuple. Pada sisi SQL, arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5) menghasilkan urutan yang sama dan dicetak melalui itertuples(index=False, name=None). Terakhir, lima zona tujuan terpopuler dihitung dari G.in_degree(weight="jumlah") yang diurutkan sorted() dengan key berupa *lambda* bernilai negatif bobot lalu nomor zona, dan di-*assign* ke populer.

Keputusan pra-pemrosesan pada sel ini, urutan lima arus terbesar diberi pengurut tambahan berupa nomor asal dan tujuan sesuai bagian M.5 modul, karena dua arus dapat memiliki jumlah perjalanan yang sama persis sehingga urutannya tidak terjamin dan perbandingan graph dengan SQL dapat tampak berbeda tanpa sebab yang nyata.

Dugaan: jumlah node diperkirakan mendekati 263 zona yang ada pada kamus zona TLC, sedangkan jumlah edge diperkirakan jauh lebih banyak karena hampir setiap pasangan zona pernah dilalui, dan kedua daftar lima arus terbesar diperkirakan sama persis.

In [ ]:
import networkx as nx

arus = ukur("hitung arus antar zona", lambda: con_d.sql("""SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
          FROM perjalanan GROUP BY 1, 2""").df())

G = nx.DiGraph()
for r in arus.itertuples(index=False):
  G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
print("node (zona):", G.number_of_nodes(), "| edge (arus):", G.number_of_edges())

top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a,b,d["jumlah"]) for a, b, d in top_graf])
top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL :", list(top_sql.itertuples(index=False, name=None)))

populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler: ", populer)


[hitung arus antar zona] 0.2048 s
node (zona): 262 | edge (arus): 21583
graph: [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
SQL : [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
zona tujuan terpopuler:  [(236, 144209), (237, 130369), (161, 113991), (230, 87706), (170, 87110)]


## K-5. Zona yang Dicapai dalam Dua Langkah

Konstanta MIN ditetapkan 50 dan zona_awal diambil dari populer[0][0], yaitu zona tujuan terpopuler. Cara graph diwakili fungsi dua_langkah_graph() yang menyiapkan set kosong hasil, menjalankan *for loop* atas G[zona_awal].items(), melewati tetangga yang bobot jumlah-nya kurang dari MIN melalui pernyataan continue, lalu pada *for loop* bersarang atas G[b].items() menambahkan simpul c ke hasil apabila bobotnya tidak kurang dari MIN dan c bukan zona_awal. Cara SQL diwakili dua_langkah_sql() setelah arus didaftarkan ke DuckDB melalui con_d.register("arus", arus), yaitu con_d.sql() menjalankan SELECT DISTINCT a2.tujuan atas arus a1 yang di-JOIN dengan arus a2 pada a1.tujuan sama dengan a2.asal, disaring pada a1.asal sama dengan zona_awal, kedua bobot tidak kurang dari MIN, dan a2.tujuan bukan zona_awal, lalu hasilnya diubah ke set dari kolom tujuan DataFrame.

Keduanya dijalankan di dalam ukur(), jumlah zona terjangkau dicetak dengan len() dan kesamaan hasil diperiksa dengan operator ==. Setelah itu graph tak berbobot H dibentuk dari nx.DiGraph() atas *list comprehension* yang hanya memuat edge berbobot tidak kurang dari MIN, lalu *for loop* atas nilai k 1 sampai 4 mencetak jumlah zona hasil nx.single_source_shortest_path_length(H, zona_awal, cutoff=k) yang dikurangi satu agar zona_awal tidak ikut dihitung.

Keputusan pra-pemrosesan pada sel ini, arus dengan kurang dari 50 perjalanan diabaikan mengikuti modul agar jalur yang hanya muncul sekali tidak dianggap jalur yang biasa dilalui, dengan konsekuensi jangkauan yang terhitung lebih kecil daripada jangkauan sebenarnya pada data penuh.

Dugaan: cara graph diperkirakan beberapa kali lebih cepat daripada SQL *self-join* karena penelusuran tetangga tidak perlu menggabungkan tabel dengan dirinya sendiri, sedangkan jumlah zona terjangkau diperkirakan melonjak dari k sama dengan 1 ke k sama dengan 2 lalu berhenti bertambah pada k berikutnya.

In [ ]:
MIN = 50
zona_awal = populer[0][0]

def dua_langkah_graph():
  hasil = set()
  for b, d1 in G[zona_awal].items():
    if d1["jumlah"] < MIN:
      continue
    for c, d2 in G[b].items():
      if d2["jumlah"] >= MIN and c != zona_awal:
        hasil.add(c)
  return hasil


con_d.register("arus", arus)
def dua_langkah_sql():
  return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
          AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])
hg = ukur("2 langkah - graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah - SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal: ", zona_awal, "| terjangkau (graph):", len(hg), "| (SQL):", len(hs), "| sama?",
      hg == hs)

H = nx.DiGraph([(a,b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1,2,3,4):
  n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
  print(f"dalam <= {k} langkah: {n} zona")

[2 langkah - graph (NetworkX)] 0.0082 s
[2 langkah - SQL self-join (DuckDB)] 0.0088 s
zona awal:  236 | terjangkau (graph): 214 | (SQL): 214 | sama? True
dalam <= 1 langkah: 74 zona
dalam <= 2 langkah: 214 zona
dalam <= 3 langkah: 214 zona
dalam <= 4 langkah: 214 zona


## K-6. Indeks Spasial

Kelas STRtree beserta fungsi box dan points diimpor dari shapely. Pembangkit acak dibentuk dengan np.random.default_rng(42), lalu N ditetapkan 200.000 dan koordinat tiruan dibentuk dari rng.normal() yang dibatasi clip(), yaitu bujur pada rentang -74,25 sampai -73,70 dan lintang pada rentang 40,50 sampai 40,92, kemudian keduanya diubah menjadi objek geometri melalui points(bujur, lintang). Wilayah pencarian dibentuk dari M bernilai 1000 titik pusat hasil rng.uniform() pada cx dan cy, lalu kotak disusun dengan *list comprehension* atas zip(cx, cy) yang memanggil box() dengan batas digeser 0,005 derajat ke setiap arah.

Tiga cara dibandingkan. Fungsi cara_loop() me-*return* *list comprehension* yang memanggil sum() atas pemeriksaan k.contains(p) untuk setiap titik, dan hanya dijalankan pada lima kotak pertama melalui *slicing*. Fungsi cara_numpy() menjalankan *for loop* atas zip(cx, cy), menyusun *boolean mask* m dari empat perbandingan atas bujur dan lintang yang digabung operator &, lalu menambahkan int(m.sum()) ke list hasil. Cara ketiga membangun indeks dengan STRtree(titik) di dalam ukur() dan mencarinya melalui *list comprehension* atas kotak yang memanggil pohon.query(k, predicate="contains") lalu len(). Hasil lima kotak pertama kedua cara dicetak berdampingan, kesamaan NumPy dengan STRtree dan kesamaan lima hasil loop dengan potongan h_idx[:5] diperiksa dengan operator ==, dan total titik yang ditemukan dijumlahkan dengan sum().

Keputusan pra-pemrosesan pada sel ini, titik jemput dibangkitkan sebagai data tiruan dengan *seed* 42 karena data TLC sejak 2016 hanya memuat nomor zona tanpa koordinat lintang-bujur, dengan konsekuensi angka K-6 menggambarkan cara kerja indeks dan bukan pola perjalanan taksi yang sebenarnya, sebagaimana diingatkan modul pada bagian I.

Dugaan: loop Python diperkirakan ribuan kali lebih lambat daripada STRtree sehingga seribu kotak tidak mungkin dijalankan, dan STRtree diperkirakan tetap menang telak atas NumPy dengan selisih ratusan kali karena hanya kelompok kotak yang bersinggungan yang benar-benar diperiksa.

In [ ]:
from shapely import STRtree, box, points

rng = np.random.default_rng(42)
N = 200_000
bujur = rng.normal(-73.97, .06, N).clip(-74.25, -73.70)
lintang = rng.normal(40.74, .05, N).clip(40.50, 40.92)
titik = points(bujur, lintang)

M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)

kotak = [box(x - .005, y - .005, x + .005, y + .005) for x, y in zip(cx, cy)]

def cara_loop():
  return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

def cara_numpy():
  hasil = []
  for x, y in zip(cx, cy):
    m = (bujur >= x -.005) & (bujur <= x + .005) & (lintang >= y - .005) & (lintang <= y + .005)
    hasil.append(int(m.sum()))
  return hasil

h_loop = ukur("cara 1 - loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 - NumPy, 1000 kotak", cara_numpy)

pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 - STRtree, 1000 kotak",
             lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama - loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?",
      h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))


[cara 1 - loop Python, 5 kotak] 14.6823 s
[cara 2 - NumPy, 1000 kotak] 0.5619 s
[bangun indeks STRtree] 0.0715 s
[cara 3 - STRtree, 1000 kotak] 0.2409 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


## K-7. Delta Lake: Versi, Time Travel, dan Schema Enforcement

Kelas DeltaTable dan fungsi write_deltalake diimpor dari deltalake. Tujuh kolom ditetapkan pada KOLOM_DELTA, lalu data_delta dibentuk dari df[KOLOM_DELTA].sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True). Lokasi tabel PATH_DELTA disusun dari f-string atas DIR_SIMPAN seperti pada modul, dan sisa percobaan sebelumnya dihapus dengan shutil.rmtree() apabila os.path.exists() bernilai True sehingga sel aman dijalankan ulang.

Penulisan versi 0 dari data_delta.iloc[:100_000] melalui write_deltalake() bermode overwrite dibungkus blok *try-except* agar galat sistem berkas tertangkap dan tidak menghentikan Run all, yaitu *exception* OSError dicetak bersama nama kelasnya melalui type(e).\_\_name\_\_ pada blok *except*, lalu sisa penulisan yang terlanjur tertulis dihapus melalui shutil.rmtree() berargumen ignore_errors=True sehingga folder tidak lengkap tidak tertinggal di Google Drive.

Keputusan pra-pemrosesan pada sel ini, koreksi ditetapkan hanya pada perjalanan Tunai di wilayah Bronx dengan penambahan 1,0 mengikuti modul, karena jumlah baris yang terkena dapat dihitung lebih dahulu sehingga selisih total_bayar antarversi dapat diperiksa kebenarannya pada sel berikutnya.

Dugaan: penulisan diperkirakan berhasil sebagaimana contoh keluaran modul, karena Google Drive terpasang sebagai folder biasa sehingga pustaka deltalake diperkirakan memperlakukannya seperti disk mana pun.

In [ ]:
from deltalake import DeltaTable, write_deltalake

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)),
random_state=42).reset_index(drop=True)

PATH_DELTA = f"{DIR_SIMPAN}/trips_delta"
if os.path.exists(PATH_DELTA):
  shutil.rmtree(PATH_DELTA)

try:
  write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
  dt = DeltaTable(PATH_DELTA)
  print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))
except OSError as e:
  print("GAGAL menulis tabel Delta di Google Drive")
  print(type(e).__name__, ":", str(e).replace("\n", " "))
  shutil.rmtree(PATH_DELTA, ignore_errors=True)

GAGAL menulis tabel Delta di Google Drive
OSError : Generic LocalFileSystem error           ↳ Unable to rename file            ↳ Operation not permitted (os error 1) 


## K-7. Catatan Error

Penulisan tabel Delta ke Google Drive berhenti dengan OSError berbunyi Generic LocalFileSystem error, Unable to rename file, Operation not permitted. Penyebabnya, write_deltalake() menyelesaikan setiap komit dengan membuat tautan berkas lalu memindahkannya secara atomik di dalam _delta_log, sedangkan Google Drive yang dipasang melalui FUSE tidak mengizinkan operasi tersebut. Galat ini berasal dari kemampuan sistem berkas, bukan dari isi data maupun dari kode modul, sehingga tidak dapat dihindari selama PATH_DELTA menunjuk ke folder Drive.

Solusinya, tabel Delta dibuat di disk runtime pada /content/trips_delta, yaitu sistem berkas biasa yang mendukung pemindahan atomik, lalu dipindahkan ke Google Drive dalam bentuk arsip zip pada sel terakhir notebook sesuai daftar berkas yang diminta modul. Seluruh baris berikutnya sama persis dengan modul dan hanya nilai PATH_DELTA yang berbeda.

## K-7. Solusi Terhadap Error

Konstanta PATH_DELTA ditetapkan ulang pada /content/trips_delta dan sisa percobaan sebelumnya dihapus dengan shutil.rmtree() apabila os.path.exists() bernilai True. Tiga versi kemudian dibentuk berurutan seperti pada modul. Versi 0 ditulis dari data_delta.iloc[:100_000] melalui write_deltalake() bermode overwrite, lalu dt dibentuk dari DeltaTable(PATH_DELTA) dan versi beserta jumlah barisnya dicetak dengan dt.version() dan len(dt.to_pandas()). Versi 1 menambahkan data_delta.iloc[100_000:150_000] bermode append dan dicetak dengan cara yang sama. Versi 2 berupa koreksi, yaitu sekarang di-*assign* dari data_delta.iloc[:150_000], jumlah baris yang akan dikoreksi dihitung dari dua *boolean mask* atas wilayah_jemput bernilai Bronx dan metode_bayar bernilai Tunai yang digabung operator & lalu dijumlahkan sum() dan di-*cast* int(), kemudian dt.update() dijalankan dengan predicate pada kedua kolom itu dan updates yang menambah total_bayar sebesar 1.0, lalu nomor versi terbaru dicetak.

Dugaan: nomor versi diperkirakan naik dari 0 ke 1 lalu ke 2, jumlah baris diperkirakan 100.000 lalu 150.000 dan tetap 150.000 setelah update, sedangkan selisih total_bayar antara versi 2 dan versi 1 diperkirakan sama persis dengan jumlah baris yang dikoreksi karena setiap baris ditambah 1,0.

In [ ]:
PATH_DELTA = "/content/trips_delta"
if os.path.exists(PATH_DELTA):
  shutil.rmtree(PATH_DELTA)


write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))


write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == 'Bronx') & (sekarang["metode_bayar"] ==
                                                            "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)
dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates = {"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA).version())

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 20
versi: 2


## K-7. Time Travel, Riwayat, dan Penegakan Skema

Tiga keadaan tabel dibaca melalui DeltaTable(PATH_DELTA, version=0).to_pandas(), DeltaTable(PATH_DELTA, version=1).to_pandas(), dan DeltaTable(PATH_DELTA).to_pandas() lalu di-*assign* ke v0, v1, dan terbaru, kemudian jumlah barisnya dicetak dengan len(). Selisih di-*assign* dari pengurangan terbaru["total_bayar"].sum() dengan v1["total_bayar"].sum(), dicetak setelah dibulatkan round(selisih, 2) bersama n_koreksi sebagai pembanding. Riwayat perubahan ditampilkan melalui *for loop* atas DeltaTable(PATH_DELTA).history() yang mencetak h.get("version") dan h.get("operation"), lalu isi folder _delta_log dicetak melalui sorted(os.listdir()) sehingga satu berkas JSON per versi terlihat.

Penegakan skema diuji pada bagian terakhir, yaitu tambahan di-*assign* dari data_delta.iloc[150_000:150_010].copy() dan diberi kolom_asing bernilai 1. Di dalam blok *try*, write_deltalake() bermode append dipanggil dan keterangan BERHASIL dicetak apabila tidak ada galat, sedangkan pada blok *except* seluruh Exception ditangkap dan nama kelasnya dicetak melalui type(e).\_\_name\_\_. Nomor versi setelah percobaan itu dicetak kembali untuk memperlihatkan bahwa penulisan yang ditolak tidak meninggalkan versi baru.

Dugaan: jumlah baris versi 0 dan versi 1 diperkirakan berbeda sedangkan versi 1 dan versi terbaru diperkirakan sama, penulisan berkolom asing diperkirakan ditolak dengan SchemaMismatchError, dan nomor versi diperkirakan tetap 2 setelah penolakan itu.

In [ ]:
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()
print("baris v0:", len(v0), "| v1:", len(v1), "| terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), "| jumlah baris dikoreksi:", n_koreksi)

for h in DeltaTable(PATH_DELTA).history():
  print(h.get("version"), h.get("operation"))

print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
  write_deltalake(PATH_DELTA, tambahan, mode="append")
  print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
  print("append kolom asing DITOLAK:", type(e).__name__)
print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA).version())

baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 20.0 | jumlah baris dikoreksi: 20
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


## K-8. Small File Problem pada Tabel Delta

Objek dt dibentuk dari DeltaTable(PATH_DELTA), lalu jumlah file data aktif dicetak dengan len(dt.file_uris()) bersama dt.version() sebagai keadaan awal. Data yang masuk sedikit demi sedikit ditiru dengan *for loop* atas range(20) yang pada setiap iterasi memotong 500 baris melalui data_delta.iloc[] dengan batas dihitung dari indeks i, lalu menuliskannya dengan write_deltalake() bermode append, sehingga setiap penulisan menghasilkan satu berkas Parquet baru. Keadaan setelah dua puluh penambahan dicetak kembali, yaitu jumlah file aktif, nomor versi, dan jumlah baris. Waktu baca sebelum compaction diukur dengan time.perf_counter() yang mengapit DeltaTable(PATH_DELTA).to_pandas() dan di-*assign* ke t_sebelum.

Berkas kecil digabung melalui dt.optimize.compact(), lalu hasilnya dicetak sebagai *dict comprehension* yang mengambil kunci numFilesAdded dan numFilesRemoved. Objek dt dibentuk ulang agar membaca catatan terbaru, waktu baca diukur kembali ke t_sesudah dengan cara yang sama, lalu jumlah file aktif, nomor versi, jumlah baris, dan kedua waktu baca dicetak melalui f-string berformat .4f. Bagian akhir memperlihatkan bahwa berkas lama belum terhapus, yaitu jumlah berkas Parquet di disk dihitung dengan len(glob.glob()) dan daftar berkas yang dapat dihapus diambil dari dt.vacuum() berargumen retention_hours=0, enforce_retention_duration=False, dan dry_run=True.

Keputusan pra-pemrosesan pada sel ini, VACUUM dijalankan dengan dry_run=True sehingga hanya melaporkan tanpa menghapus, karena retention_hours=0 pada tabel sungguhan akan menghapus berkas yang masih dibutuhkan pembaca lain dan memutus kemampuan time travel, sebagaimana diperingatkan modul setelah K-8.

Dugaan: jumlah file aktif diperkirakan naik menjadi sekitar 21 setelah dua puluh penambahan kecil lalu turun menjadi satu setelah compaction dengan jumlah baris tetap, dan waktu baca sesudah compaction diperkirakan beberapa kali lebih cepat daripada sebelumnya karena berkas yang harus dibuka tinggal satu.

In [ ]:
dt = DeltaTable(PATH_DELTA)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

for i in range(20):
  potongan = data_delta.iloc[150_000 + i * 500: 150_000 + ( i + 1) * 500]
  write_deltalake(PATH_DELTA, potongan, mode="append")
dt = DeltaTable(PATH_DELTA)
print("file data aktif setelah 20 append kecil:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))

t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sebelum = time.perf_counter() - t0

hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})

dt = DeltaTable(PATH_DELTA)
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sesudah = time.perf_counter() - t0
print("file data aktif setelah compact:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA}/*.parquet")))
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True)
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.0879 s | sesudah: 0.0515 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


## K-9. Merangkum dan Memilih

Seluruh hasil pengukuran pada list catatan diubah menjadi DataFrame rekap melalui pd.DataFrame(catatan), disimpan ke DIR_SIMPAN sebagai benchmark_nosql.csv dengan to_csv() berargumen index=False, lalu ditampilkan sebagai teks tanpa kolom indeks melalui rekap.to_string(index=False).

Dugaan: tabel rekap diperkirakan memuat sekitar dua puluh baris, yaitu satu baris pembacaan data, tujuh baris K-2, empat baris K-4, tiga baris K-5, dan empat baris K-6, sedangkan K-7 dan K-8 diperkirakan tidak menambah baris karena waktunya diukur langsung dengan time.perf_counter().

In [ ]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                            langkah   detik
                                  baca trips_bersih  1.9129
                             relasional: muat tabel  3.4134
                             key-value: bangun dict 10.0131
             cari 200 perjalanan - SQL tanpa indeks  8.1600
            cari 200 perjalanan - SQL dengan indeks  0.0040
                    cari 200 perjalanan - key-value  0.0013
                  hitung per wilayah - SQL GROUP BY  0.2040
  hitung per wilayah - key-value (buka semua nilai)  1.4833
DuckDB - hitung per wilayah (langsung dari Parquet)  0.0965
               SQLite - muat seluruh baris ke tabel  9.2538
                        SQLite - hitung per wilayah  2.6813
               DuckDB - hanya wilayah Staten Island  0.0187
                             hitung arus antar zona  0.2048
                       2 langkah - graph (NetworkX)  0.0082
                 2 langkah - SQL self-join (DuckDB)  0.0088
                      cara 1 - loop Pyth

## K-9. Tabel Rekap Pilihan Penyimpanan

Series waktu dibentuk dari pd.DataFrame(catatan) yang dirapikan drop_duplicates("langkah", keep="last"), diberi set_index("langkah"), lalu diambil kolom detik-nya, sehingga durasi tiap langkah dapat dipanggil melalui nama labelnya. Didefinisikan fungsi susun_rekap_pilihan(waktu) yang me-*return* DataFrame dari list berisi enam dict, yaitu satu dict untuk tiap baris tabel rekap pada modul dengan kunci kebutuhan, cara_unggul, cara_lemah, dan angka. Nilai pada kunci angka dibentuk melalui f-string yang membagi dua label pengukuran secara langsung dan diformat ,.0f atau ,.1f, sedangkan dua baris yang tidak diukur diisi keterangan kualitatif. Fungsi dipanggil dengan Series waktu, hasilnya disimpan ke DIR_SIMPAN sebagai rekap_pilihan_penyimpanan.csv dengan index=False, lalu ditampilkan sebagai keluaran sel.

Dugaan: keempat baris yang berangka diperkirakan searah dengan kesimpulan modul, yaitu key-value dan indeks menang telak untuk pengambilan satu data, penyimpanan per kolom menang untuk perhitungan atas banyak baris, graph menang tipis atau seri untuk keterhubungan, dan indeks spasial menang beberapa kali lipat terhadap pemeriksaan seluruh titik.

In [ ]:
waktu = (pd.DataFrame(catatan)
         .drop_duplicates("langkah", keep="last")
         .set_index("langkah")["detik"])

def susun_rekap_pilihan(waktu):
  return pd.DataFrame([
      {"kebutuhan": "Ambil satu data lewat id",
       "cara_unggul": "Key-value; tabel dengan indeks",
       "cara_lemah": "Tabel tanpa indeks",
       "angka": f"{waktu['cari 200 perjalanan - SQL tanpa indeks'] / waktu['cari 200 perjalanan - key-value']:,.0f} kali"},
      {"kebutuhan": "Menghitung atau merangkum banyak baris",
       "cara_unggul": "Per kolom (DuckDB pada Parquet)",
       "cara_lemah": "Key-value (buka semua nilai)",
       "angka": f"key-value {waktu['hitung per wilayah - key-value (buka semua nilai)'] / waktu['hitung per wilayah - SQL GROUP BY']:,.1f} kali SQL; "
                f"SQLite {waktu['SQLite - hitung per wilayah'] / waktu['DuckDB - hitung per wilayah (langsung dari Parquet)']:,.1f} kali DuckDB"},
      {"kebutuhan": "Data dengan field opsional atau berubah-ubah",
       "cara_unggul": "Document",
       "cara_lemah": "Tabel (kolom NULL, migrasi)",
       "angka": "kualitatif: field cuaca masuk tanpa ALTER TABLE"},
      {"kebutuhan": "Keterhubungan dengan jumlah langkah bebas",
       "cara_unggul": "Graph",
       "cara_lemah": "SQL (satu JOIN per langkah)",
       "angka": f"{waktu['2 langkah - SQL self-join (DuckDB)'] / waktu['2 langkah - graph (NetworkX)']:,.1f} kali pada dua langkah"},
      {"kebutuhan": "Mencari lokasi dalam wilayah",
       "cara_unggul": "Spasial dengan indeks",
       "cara_lemah": "Memeriksa semua titik",
       "angka": f"{waktu['cara 2 - NumPy, 1000 kotak'] / waktu['cara 3 - STRtree, 1000 kotak']:,.1f} kali terhadap NumPy"},
      {"kebutuhan": "Perubahan yang harus aman dan bisa dibatalkan",
       "cara_unggul": "Tabel Delta",
       "cara_lemah": "Folder Parquet biasa",
       "angka": "kualitatif: time travel, penolakan skema, dan restore"},
  ])

rekap_pilihan = susun_rekap_pilihan(waktu)
rekap_pilihan.to_csv(f"{DIR_SIMPAN}/rekap_pilihan_penyimpanan.csv", index=False)
rekap_pilihan

,kebutuhan,cara_unggul,cara_lemah,angka
0,Ambil satu data lewat id,Key-value; tabel dengan indeks,Tabel tanpa indeks,"6,277 kali"
1,Menghitung atau merangkum banyak baris,Per kolom (DuckDB pada Parquet),Key-value (buka semua nilai),key-value 7.3 kali SQL; SQLite 27.8 kali DuckDB
2,Data dengan field opsional atau berubah-ubah,Document,"Tabel (kolom NULL, migrasi)",kualitatif: field cuaca masuk tanpa ALTER TABLE
3,Keterhubungan dengan jumlah langkah bebas,Graph,SQL (satu JOIN per langkah),1.1 kali pada dua langkah
4,Mencari lokasi dalam wilayah,Spasial dengan indeks,Memeriksa semua titik,2.3 kali terhadap NumPy
5,Perubahan yang harus aman dan bisa dibatalkan,Tabel Delta,Folder Parquet biasa,"kualitatif: time travel, penolakan skema, dan ..."


# O. Analisis Hasil

## O-1. Biaya Pencarian Satu Data

Series waktu yang sudah dibentuk pada K-9 dipakai kembali sebagai sumber angka, sehingga durasi tiap langkah dapat diambil melalui nama labelnya. Tiga durasi pencarian 200 id diambil dengan pengindeksan label, yaitu cari 200 perjalanan - SQL tanpa indeks, cari 200 perjalanan - SQL dengan indeks, dan cari 200 perjalanan - key-value, masing-masing di-*assign* ke tanpa_indeks, dengan_indeks, dan kunci_nilai. Biaya satu pencarian dihitung dengan membagi tiap durasi dengan 200 lalu mengalikannya dengan 1e6 sehingga satuannya menjadi mikrodetik, sedangkan dua rasio dihitung dari pembagian tanpa_indeks terhadap dengan_indeks dan dengan_indeks terhadap kunci_nilai. Seluruh angka dicetak melalui print() dengan f-string berformat .4f untuk detik serta ,.1f untuk mikrodetik dan rasio.

Dugaan: tabel tanpa indeks diperkirakan ratusan kali lebih lambat daripada tabel berindeks karena setiap pencarian memindai baris demi baris, sedangkan tabel berindeks diperkirakan hanya beberapa kali lebih lambat daripada key-value karena keduanya menuju baris melalui struktur indeks, sehingga selisih yang benar-benar terasa bagi pengguna hanya muncul pada pasangan pertama.

In [ ]:
tanpa_indeks = waktu["cari 200 perjalanan - SQL tanpa indeks"]
dengan_indeks = waktu["cari 200 perjalanan - SQL dengan indeks"]
kunci_nilai = waktu["cari 200 perjalanan - key-value"]

print(f"tanpa indeks  : {tanpa_indeks:.4f} s | {tanpa_indeks / 200 * 1e6:>9,.1f} mikrodetik per pencarian")
print(f"dengan indeks : {dengan_indeks:.4f} s | {dengan_indeks / 200 * 1e6:>9,.1f} mikrodetik per pencarian")
print(f"key-value     : {kunci_nilai:.4f} s | {kunci_nilai / 200 * 1e6:>9,.1f} mikrodetik per pencarian")
print(f"tanpa indeks  : {tanpa_indeks / dengan_indeks:,.1f} kali tabel berindeks")
print(f"tabel indeks  : {dengan_indeks / kunci_nilai:,.1f} kali key-value")

tanpa indeks  : 8.1600 s |  40,800.0 mikrodetik per pencarian
dengan indeks : 0.0040 s |      20.0 mikrodetik per pencarian
key-value     : 0.0013 s |       6.5 mikrodetik per pencarian
tanpa indeks  : 2,040.0 kali tabel berindeks
tabel indeks  : 3.1 kali key-value


## Jawaban O-1

Tabel tanpa indeks lebih lambat [isi dari keluaran] kali daripada tabel berindeks, sedangkan tabel berindeks lebih lambat [isi dari keluaran] kali daripada key-value. Penyebabnya, tanpa indeks SQLite memeriksa baris demi baris sampai id yang diminta ditemukan, sedangkan indeks idx_id yang dibuat melalui CREATE INDEX membuat pencarian hanya menelusuri beberapa simpul, dan dict pada kv menghitung letak nilai langsung dari kuncinya. Perbedaan yang benar-benar berarti hanya pada pasangan pertama, karena selisihnya [isi dari keluaran] mikrodetik per pencarian, sementara selisih pasangan kedua hanya [isi dari keluaran] mikrodetik sehingga tidak terasa pada satu permintaan pengguna.

## O-2. Harga Kesederhanaan Key-value

Dua durasi perhitungan rata-rata per wilayah diambil dari Series waktu, yaitu hitung per wilayah - SQL GROUP BY dan hitung per wilayah - key-value (buka semua nilai), lalu di-*assign* ke sql_group dan kv_buka. Rasio kv_buka terhadap sql_group dihitung untuk menyatakan berapa kali key-value lebih lambat, jumlah nilai yang harus dibuka diambil dari len(kv), dan biaya membuka satu nilai dihitung dengan membagi kv_buka dengan len(kv) lalu mengalikannya dengan 1e6. Keempat angka dicetak melalui print() dengan pemisah ribuan pada len(kv) dan dua angka desimal pada biaya per nilai.

Dugaan: key-value diperkirakan sekitar lima sampai sepuluh kali lebih lambat karena hitung_kv() memanggil json.loads() untuk setiap nilai sebelum field total_bayar dapat dijumlahkan, sedangkan hitung_sql() menyerahkan penjumlahan kepada SQLite yang sudah mengetahui tipe kolomnya.

In [ ]:
sql_group = waktu["hitung per wilayah - SQL GROUP BY"]
kv_buka = waktu["hitung per wilayah - key-value (buka semua nilai)"]

print(f"SQL GROUP BY : {sql_group:.4f} s")
print(f"key-value    : {kv_buka:.4f} s -> {kv_buka / sql_group:,.1f} kali lebih lambat")
print(f"nilai JSON yang dibuka json.loads(): {len(kv):,}")
print(f"biaya membuka satu nilai           : {kv_buka / len(kv) * 1e6:,.2f} mikrodetik")

SQL GROUP BY : 0.2040 s
key-value    : 1.4833 s -> 7.3 kali lebih lambat
nilai JSON yang dibuka json.loads(): 300,000
biaya membuka satu nilai           : 4.94 mikrodetik


## Jawaban O-2

Key-value lebih lambat [isi dari keluaran] kali daripada SQL GROUP BY untuk pertanyaan yang sama. Sebabnya dijelaskan bagian M.2 modul, yaitu nilai pada key-value hanyalah sepotong teks bagi penyimpannya, sehingga hitung_kv() harus membuka seluruh [isi dari keluaran] nilai dengan json.loads(), mengambil field wilayah_jemput dan total_bayar, lalu menjumlahkannya sendiri di dalam dict total dan jumlah. SQLite mengetahui total_bayar sebagai kolom angka sehingga penjumlahan dikerjakan langsung oleh mesin basis data tanpa tahap penguraian. Harga kesederhanaan key-value adalah hilangnya pengetahuan tentang isi nilai, dan harga itu baru terasa pada pertanyaan yang menyentuh semua data, bukan pada pengambilan satu kunci.

## O-3. Keuntungan dan Risiko Dokumen Berfield Bebas

Pertanyaan ini tidak memerlukan kode baru karena buktinya sudah ada pada keluaran K-3, yaitu jumlah dokumen yang punya dan tidak punya field tip, serta dokumen tambahan yang membawa field cuaca.

Keuntungannya, dokumen baru dengan field cuaca dapat dimasukkan melalui tabel.insert() tanpa perintah ALTER TABLE, tanpa migrasi, dan tanpa menyentuh 20.000 dokumen yang sudah ada, sehingga perubahan bentuk data tidak menghentikan layanan. Risikonya, TinyDB tidak memeriksa nama field, sehingga salah ketik seperti biya atau cuacaa tetap diterima dan baru ketahuan ketika Q.biaya.total dipakai dan hasil pencariannya kosong, sehingga beban validasi berpindah ke aplikasi. Perbandingannya ada pada K-7, yaitu tabel Delta justru menolak kolom asing melalui SchemaMismatchError dan nomor versinya tidak berubah.

## O-4. Keuntungan Penyimpanan per Kolom

Empat durasi diambil dari Series waktu, yaitu DuckDB - hitung per wilayah (langsung dari Parquet), SQLite - muat seluruh baris ke tabel, SQLite - hitung per wilayah, dan DuckDB - hanya wilayah Staten Island, lalu di-*assign* ke duck_hitung, sqlite_muat, sqlite_hitung, dan duck_filter. Dua rasio dihitung, yaitu sqlite_hitung terhadap duck_hitung untuk membandingkan perhitungannya saja dan penjumlahan sqlite_muat dengan sqlite_hitung terhadap duck_hitung untuk membandingkan seluruh biayanya. Rasio duck_hitung terhadap duck_filter dihitung sebagai ukuran manfaat *partition pruning*, sedangkan jumlah kolom yang disentuh dicetak dari angka 2 untuk kolom wilayah_jemput dan total_bayar, df.shape[1] untuk kolom data, dan len(KOLOM_S) untuk kolom yang disalin ke SQLite.

Dugaan: DuckDB diperkirakan sepuluh sampai lima belas kali lebih cepat pada perhitungannya saja dan jauh lebih cepat lagi bila tahap muat SQLite ikut dihitung, sedangkan penyaringan satu wilayah diperkirakan beberapa kali lebih cepat daripada perhitungan seluruh wilayah.

In [ ]:
duck_hitung = waktu["DuckDB - hitung per wilayah (langsung dari Parquet)"]
sqlite_muat = waktu["SQLite - muat seluruh baris ke tabel"]
sqlite_hitung = waktu["SQLite - hitung per wilayah"]
duck_filter = waktu["DuckDB - hanya wilayah Staten Island"]

print(f"DuckDB  : {duck_hitung:.4f} s")
print(f"SQLite  : {sqlite_hitung:.4f} s menghitung + {sqlite_muat:.4f} s memuat tabel")
print(f"perhitungan saja    : {sqlite_hitung / duck_hitung:,.1f} kali")
print(f"termasuk tahap muat : {(sqlite_muat + sqlite_hitung) / duck_hitung:,.1f} kali")
print(f"filter satu wilayah : {duck_filter:.4f} s -> {duck_hitung / duck_filter:,.1f} kali lebih cepat daripada seluruh wilayah")
print("kolom yang disentuh query:", 2, "dari", df.shape[1], "kolom data |", len(KOLOM_S), "kolom disalin ke SQLite")

DuckDB  : 0.0965 s
SQLite  : 2.6813 s menghitung + 9.2538 s memuat tabel
perhitungan saja    : 27.8 kali
termasuk tahap muat : 123.7 kali
filter satu wilayah : 0.0187 s -> 5.2 kali lebih cepat daripada seluruh wilayah
kolom yang disentuh query: 2 dari 18 kolom data | 5 kolom disalin ke SQLite


## Jawaban O-4

DuckDB lebih cepat [isi dari keluaran] kali pada perhitungannya saja dan [isi dari keluaran] kali apabila tahap muat SQLite ikut dihitung. Penyebab pertama adalah *column pruning*, yaitu query hanya menyentuh dua kolom dari seluruh kolom Parquet, sedangkan SQLite harus membaca kelima kolom KOLOM_S yang sudah disalin ke berkas baris.db. Penyebab kedua bukan soal format kolom, yaitu DuckDB memproses banyak baris sekaligus secara *vectorized* dan memakai beberapa inti prosesor, serta tidak melalui tahap muat sama sekali karena read_parquet() membaca berkasnya langsung di tempat. Penyaringan pada kolom partisi menambah keuntungan ketiga, yaitu *partition pruning* membuat query Staten Island [isi dari keluaran] kali lebih cepat karena hanya folder wilayah itu yang dibuka.

## O-5. Alasan Graph Tetap Layak Dipilih

Dua durasi dua langkah dicetak dari Series waktu, yaitu 2 langkah - graph (NetworkX) dan 2 langkah - SQL self-join (DuckDB). Setelah itu dijalankan *for loop* atas nilai k 1 sampai 4 yang memanggil nx.single_source_shortest_path_length(H, zona_awal, cutoff=k), lalu len() atas hasilnya dikurangi 1 untuk membuang zona_awal sendiri dari hitungan. Setiap baris keluaran disertai keterangan bahwa pada graph nilai k cukup diganti pada argumen cutoff, sedangkan pada SQL setiap langkah tambahan menuntut satu *self-join* tambahan.

Dugaan: jumlah zona terjangkau diperkirakan naik tajam dari k sama dengan 1 ke k sama dengan 2, lalu hampir berhenti bertambah pada k sama dengan 3 dan 4 karena jaringan zona taksi saling terhubung rapat, sehingga keunggulan graph diperkirakan terletak pada bentuk pertanyaannya, bukan pada kecepatannya.

In [ ]:
print("dua langkah - graph    :", f'{waktu["2 langkah - graph (NetworkX)"]:.4f} s')
print("dua langkah - SQL      :", f'{waktu["2 langkah - SQL self-join (DuckDB)"]:.4f} s')

for k in (1, 2, 3, 4):
  n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
  print(f"dalam <= {k} langkah: {n:>3} zona  | graph: ganti argumen cutoff | SQL: {k - 1} self-join tambahan")

dua langkah - graph    : 0.0082 s
dua langkah - SQL      : 0.0088 s
dalam <= 1 langkah:  74 zona  | graph: ganti argumen cutoff | SQL: 0 self-join tambahan
dalam <= 2 langkah: 214 zona  | graph: ganti argumen cutoff | SQL: 1 self-join tambahan
dalam <= 3 langkah: 214 zona  | graph: ganti argumen cutoff | SQL: 2 self-join tambahan
dalam <= 4 langkah: 214 zona  | graph: ganti argumen cutoff | SQL: 3 self-join tambahan


## Jawaban O-5

Pada 263 zona, dua langkah pada graph dan pada SQL sama cepatnya, sehingga alasan memilih graph tidak terletak pada kecepatan. Yang membuatnya tetap layak adalah bentuk pertanyaannya, yaitu jumlah langkah menjadi satu angka pada argumen cutoff, sedangkan pada SQL setiap langkah tambahan menuntut satu *self-join* tambahan sehingga pertanyaan tiga langkah harus ditulis ulang menjadi query yang berbeda. Hasil k sama dengan 1 sampai 4 memperlihatkan hal itu secara langsung, yaitu jangkauan melonjak dari [isi dari keluaran] zona pada satu langkah menjadi [isi dari keluaran] zona pada dua langkah, lalu berhenti bertambah pada langkah berikutnya. Graph juga menyimpan bobot jumlah pada setiap edge sehingga penyaringan MIN sebesar 50 cukup dibaca dari d1["jumlah"] dan d2["jumlah"] tanpa mengubah bentuk penyimpanan.

## O-6. Besarnya Keuntungan Indeks Spasial

Empat durasi diambil dari Series waktu, yaitu cara 1 - loop Python, 5 kotak, cara 2 - NumPy, 1000 kotak, bangun indeks STRtree, dan cara 3 - STRtree, 1000 kotak, lalu di-*assign* ke loop_5, numpy_1000, bangun_indeks, dan strtree_1000. Karena cara_loop() hanya dijalankan untuk lima kotak, biayanya untuk seribu kotak diperkirakan melalui loop_5 dibagi 5 dikali 1000 dan di-*assign* ke loop_1000. Tiga rasio dihitung, yaitu loop_1000 terhadap strtree_1000, numpy_1000 terhadap strtree_1000 untuk waktu mencari saja, dan numpy_1000 terhadap penjumlahan strtree_1000 dengan bangun_indeks agar biaya membangun indeks ikut diperhitungkan.

Dugaan: STRtree diperkirakan ribuan kali lebih cepat daripada loop Python, tetapi hanya sekitar dua sampai tiga kali lebih cepat daripada NumPy, sehingga angka percepatan yang besar diperkirakan lebih banyak menggambarkan buruknya pembanding daripada kehebatan indeksnya.

In [ ]:
loop_5 = waktu["cara 1 - loop Python, 5 kotak"]
numpy_1000 = waktu["cara 2 - NumPy, 1000 kotak"]
bangun_indeks = waktu["bangun indeks STRtree"]
strtree_1000 = waktu["cara 3 - STRtree, 1000 kotak"]
loop_1000 = loop_5 / 5 * 1000

print(f"loop Python 5 kotak    : {loop_5:.4f} s -> perkiraan 1000 kotak {loop_1000:,.1f} s")
print(f"NumPy 1000 kotak       : {numpy_1000:.4f} s")
print(f"STRtree 1000 kotak     : {strtree_1000:.4f} s (+ {bangun_indeks:.4f} s membangun indeks)")
print(f"STRtree terhadap loop  : {loop_1000 / strtree_1000:,.0f} kali")
print(f"STRtree terhadap NumPy : {numpy_1000 / strtree_1000:,.1f} kali mencari saja"
      f" | {numpy_1000 / (strtree_1000 + bangun_indeks):,.1f} kali termasuk membangun indeks")

loop Python 5 kotak    : 14.6823 s -> perkiraan 1000 kotak 2,936.5 s
NumPy 1000 kotak       : 0.5619 s
STRtree 1000 kotak     : 0.2409 s (+ 0.0715 s membangun indeks)
STRtree terhadap loop  : 12,190 kali
STRtree terhadap NumPy : 2.3 kali mencari saja | 1.8 kali termasuk membangun indeks


## Jawaban O-6

STRtree lebih cepat [isi dari keluaran] kali daripada loop Python, tetapi hanya [isi dari keluaran] kali daripada NumPy. Kedua angka itu berbeda jauh karena pembandingnya berbeda, yaitu cara_loop() memeriksa satu per satu 200.000 titik dengan k.contains(p) di dalam penerjemah Python, sedangkan cara_numpy() membandingkan seluruh titik sekaligus melalui operasi *array* pada bujur dan lintang yang dikerjakan di tingkat C. Artinya bagi cara menulis klaim percepatan, angka seperti seribu kali tidak bermakna tanpa menyebut pembandingnya, dan pembanding yang jujur adalah cara terbaik yang masuk akal, bukan cara terburuk. Biaya membangun indeks juga harus ikut disebut, karena keuntungan STRtree terhadap NumPy menyusut menjadi [isi dari keluaran] kali apabila bangun indeks STRtree dihitung sebagai bagian dari pekerjaan.

## O-7. Bukti Keandalan Tabel Delta

Dua versi tabel dibaca melalui DeltaTable(PATH_DELTA, version=1).to_pandas() dan DeltaTable(PATH_DELTA, version=2).to_pandas() lalu di-*assign* ke v1 dan v2, kemudian jumlah barisnya dicetak dengan len() berpemisah ribuan dan jumlah kolom total_bayar-nya dengan sum() yang dibulatkan dua desimal oleh round(). Selisih kedua jumlah dicetak sebagai bukti bahwa update hanya mengubah baris yang dimaksud. Setelah itu objek dt dibentuk dari DeltaTable(PATH_DELTA), riwayatnya disusun menjadi list pasangan versi dan operasi melalui *list comprehension* atas dt.history() yang memanggil h.get("version") dan h.get("operation"), jumlah berkas JSON pada _delta_log dihitung dengan len() atas *list comprehension* yang menyaring os.listdir() dengan endswith(".json"), lalu jumlah file data aktif dari len(dt.file_uris()) dibandingkan dengan jumlah berkas Parquet di disk dari len(glob.glob()).

Dugaan: jumlah baris versi 1 dan versi 2 diperkirakan sama persis sedangkan jumlah total_bayar-nya berbeda sebesar jumlah baris yang dikoreksi, riwayat diperkirakan memuat operasi WRITE, WRITE, UPDATE, dan OPTIMIZE tanpa catatan untuk penulisan yang ditolak, serta jumlah berkas Parquet di disk diperkirakan jauh lebih banyak daripada file aktif.

In [ ]:
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
v2 = DeltaTable(PATH_DELTA, version=2).to_pandas()
print(f"versi 1: {len(v1):,} baris | total_bayar {round(v1['total_bayar'].sum(), 2):,}")
print(f"versi 2: {len(v2):,} baris | total_bayar {round(v2['total_bayar'].sum(), 2):,}")
print("selisih total_bayar versi 2 - versi 1:", round(v2["total_bayar"].sum() - v1["total_bayar"].sum(), 2),
      "| jumlah baris dikoreksi:", n_koreksi)

dt = DeltaTable(PATH_DELTA)
print("riwayat (terbaru lebih dahulu):", [(h.get("version"), h.get("operation")) for h in dt.history()])
print("berkas JSON di _delta_log:", len([n for n in os.listdir(f"{PATH_DELTA}/_delta_log") if n.endswith(".json")]))
print("file data aktif:", len(dt.file_uris()), "| file parquet di disk:", len(glob.glob(f"{PATH_DELTA}/*.parquet")))

versi 1: 150,000 baris | total_bayar 4,100,172.91
versi 2: 150,000 baris | total_bayar 4,100,192.91
selisih total_bayar versi 2 - versi 1: 20.0 | jumlah baris dikoreksi: 20
riwayat (terbaru lebih dahulu): [(23, 'OPTIMIZE'), (22, 'WRITE'), (21, 'WRITE'), (20, 'WRITE'), (19, 'WRITE'), (18, 'WRITE'), (17, 'WRITE'), (16, 'WRITE'), (15, 'WRITE'), (14, 'WRITE'), (13, 'WRITE'), (12, 'WRITE'), (11, 'WRITE'), (10, 'WRITE'), (9, 'WRITE'), (8, 'WRITE'), (7, 'WRITE'), (6, 'WRITE'), (5, 'WRITE'), (4, 'WRITE'), (3, 'WRITE'), (2, 'UPDATE'), (1, 'WRITE'), (0, 'WRITE')]
berkas JSON di _delta_log: 24
file data aktif: 1 | file parquet di disk: 24


## Jawaban O-7

Pertama, update tidak menghilangkan data lama karena DeltaTable(PATH_DELTA, version=1) masih mengembalikan [isi dari keluaran] baris dengan jumlah total_bayar sebelum koreksi, sedangkan versi 2 memuat jumlah baris yang sama dengan selisih total_bayar sebesar [isi dari keluaran] yang sama dengan n_koreksi. Kedua, penulisan yang gagal tidak meninggalkan jejak karena percobaan menambah kolom_asing pada K-7 ditolak dengan SchemaMismatchError dan riwayat dt.history() tidak memuat versi untuk percobaan itu, sehingga nomor versinya tetap. Ketiga, setelah compaction berkas Parquet di disk tetap lebih banyak daripada file aktif karena file lama masih dibutuhkan untuk membaca versi sebelumnya, sebagaimana dijelaskan bagian M.6 modul, dan hanya VACUUM yang menghapusnya secara fisik.

# P. Studi Kasus

## Kasus: Rancangan Penyimpanan untuk Aplikasi Taksi

Sebuah aplikasi taksi ingin membuat lima fitur di atas data perjalanan yang sama. Pertama, halaman detail perjalanan yang dibuka penumpang dan harus terbuka dalam hitungan milidetik berdasarkan id perjalanan. Kedua, dasbor pendapatan per wilayah per jam untuk manajemen, dengan filter tanggal dan wilayah. Ketiga, fitur zona yang sering dituju dari sini dan zona yang bisa dicapai dalam dua atau tiga langkah untuk rekomendasi rute. Keempat, pencarian taksi yang pernah menjemput dalam radius 500 meter dari satu titik. Kelima, audit keuangan yang berhak meminta tabel tarif dikembalikan ke keadaan tanggal 1 apabila ada koreksi yang keliru.

Rancangan dijawab pada tiga sel berikutnya, yaitu tabel pilihan penyimpanan beserta alasan berangka, pembuktian singkat tiap fitur dengan pengukuran sendiri, dan diagram aliran data dari trips_bersih ke tiap penyimpanan.

## P-1. Tabel Rancangan Penyimpanan

Rancangan disusun sebagai list berisi lima dict yang masing-masing memuat kunci fitur, penyimpanan, alasan, dan kelemahan, lalu diubah menjadi DataFrame rancangan melalui pd.DataFrame(). Nilai pada kunci alasan dibentuk dengan f-string yang membaca Series waktu pada label pengukuran yang relevan dan menghitung rasionya langsung, sehingga angkanya berasal dari eksekusi sendiri dan ikut berubah apabila notebook dijalankan ulang. Tabel disimpan ke DIR_SIMPAN sebagai rancangan_penyimpanan.csv melalui to_csv() berargumen index=False, lalu ditampilkan sebagai keluaran sel.

Dugaan: tidak ada satu penyimpanan yang menang pada kelima fitur, sehingga rancangan diperkirakan memakai sedikitnya empat bentuk berbeda, dan fitur dasbor diperkirakan paling murah dipenuhi karena cukup memakai folder Parquet yang sudah ada tanpa menyalin data.

In [ ]:
rancangan = pd.DataFrame([
    {"fitur": "1. Detail perjalanan per id",
     "penyimpanan": "Key-value, dengan tabel berindeks sebagai cadangan",
     "alasan": f"200 pencarian key-value {waktu['cari 200 perjalanan - key-value']:.4f} s "
               f"berbanding {waktu['cari 200 perjalanan - SQL tanpa indeks']:.4f} s tanpa indeks",
     "kelemahan": "pertanyaan agregat menuntut seluruh nilai dibuka dengan json.loads()"},
    {"fitur": "2. Dasbor pendapatan per wilayah per jam",
     "penyimpanan": "Per kolom, DuckDB langsung pada Parquet berpartisi",
     "alasan": f"hitung per wilayah {waktu['DuckDB - hitung per wilayah (langsung dari Parquet)']:.4f} s "
               f"berbanding {waktu['SQLite - hitung per wilayah']:.4f} s pada SQLite, tanpa tahap muat",
     "kelemahan": "pengambilan satu baris utuh tidak secepat key-value"},
    {"fitur": "3. Rekomendasi zona dalam dua sampai tiga langkah",
     "penyimpanan": "Graph berarah dengan bobot jumlah perjalanan",
     "alasan": f"dua langkah {waktu['2 langkah - graph (NetworkX)']:.4f} s, jumlah langkah cukup "
               f"diganti pada argumen cutoff",
     "kelemahan": "merangkum data tabel besar lebih mahal daripada SQL"},
    {"fitur": "4. Taksi dalam radius 500 meter",
     "penyimpanan": "Indeks spasial STRtree di atas titik jemput",
     "alasan": f"1000 kotak {waktu['cara 3 - STRtree, 1000 kotak']:.4f} s berbanding "
               f"{waktu['cara 2 - NumPy, 1000 kotak']:.4f} s tanpa indeks",
     "kelemahan": "indeks harus dibangun ulang setiap titik berubah"},
    {"fitur": "5. Audit dan pembatalan koreksi",
     "penyimpanan": "Tabel Delta dengan time travel dan restore",
     "alasan": "versi lama tetap terbaca melalui DeltaTable(path, version=n) dan koreksi tercatat "
               "sebagai versi baru pada _delta_log",
     "kelemahan": "file lama menumpuk sampai VACUUM dijalankan"},
])

rancangan.to_csv(f"{DIR_SIMPAN}/rancangan_penyimpanan.csv", index=False)
rancangan

,fitur,penyimpanan,alasan,kelemahan
0,1. Detail perjalanan per id,"Key-value, dengan tabel berindeks sebagai cada...",200 pencarian key-value 0.0013 s berbanding 8....,pertanyaan agregat menuntut seluruh nilai dibu...
1,2. Dasbor pendapatan per wilayah per jam,"Per kolom, DuckDB langsung pada Parquet berpar...",hitung per wilayah 0.0965 s berbanding 2.6813 ...,pengambilan satu baris utuh tidak secepat key-...
2,3. Rekomendasi zona dalam dua sampai tiga langkah,Graph berarah dengan bobot jumlah perjalanan,"dua langkah 0.0082 s, jumlah langkah cukup dig...",merangkum data tabel besar lebih mahal daripad...
3,4. Taksi dalam radius 500 meter,Indeks spasial STRtree di atas titik jemput,1000 kotak 0.2409 s berbanding 0.5619 s tanpa ...,indeks harus dibangun ulang setiap titik berubah
4,5. Audit dan pembatalan koreksi,Tabel Delta dengan time travel dan restore,versi lama tetap terbaca melalui DeltaTable(pa...,file lama menumpuk sampai VACUUM dijalankan


## P-2. Pembuktian Singkat Tiap Fitur

Kelima fitur diuji dengan satu pengukuran kecil masing-masing melalui ukur() agar angkanya dapat dikutip pada laporan. Fitur pertama mengambil satu id dari id_uji dengan random.choice(), lalu membuka nilainya dari kv melalui json.loads(). Fitur kedua menjalankan SQL_DASBOR pada con_d, yaitu SELECT dengan hour(waktu_jemput), COUNT(\*), dan SUM(total_bayar) yang dibulatkan ROUND(), disaring pada wilayah_jemput Manhattan serta rentang waktu_jemput antara TANGGAL_AWAL dan TANGGAL_AKHIR, dikelompokkan GROUP BY 1, dan diubah ke DataFrame dengan df(). Fitur ketiga mengambil lima tujuan tersering dari G[zona_awal].items() melalui sorted() dengan key berupa *lambda* atas kunci jumlah. Fitur keempat membentuk pusat dengan points() lalu memanggil pohon.query() atas pusat.buffer(RADIUS_DERAJAT) dengan predicate="contains". Fitur kelima membaca DeltaTable(PATH_DELTA, version=1).to_pandas() sebagai keadaan tabel pada akhir hari pertama.

Keputusan pada sel ini, RADIUS_DERAJAT ditetapkan 0,0045 derajat karena satu derajat lintang kira-kira 111 kilometer sehingga nilai itu mewakili 500 meter, dengan konsekuensi radiusnya tidak persis pada arah bujur karena jarak satu derajat bujur menyempit di lintang New York.

Dugaan: keempat pengukuran pertama diperkirakan selesai di bawah satu detik, sedangkan pembacaan versi lama tabel Delta diperkirakan paling lama karena seluruh berkas Parquet versi itu harus dibaca kembali.

In [ ]:
id_contoh = random.choice(id_uji)
detail_perjalanan = ukur("kasus 1 - detail perjalanan dari key-value",
                           lambda: json.loads(kv[f"perjalanan:{id_contoh}"]))
print("detail id", id_contoh, "->", detail_perjalanan)

TANGGAL_AWAL, TANGGAL_AKHIR = "2023-01-08", "2023-01-15"
SQL_DASBOR = f"""SELECT hour(waktu_jemput) AS jam, COUNT(*) AS jumlah,
                        ROUND(SUM(total_bayar), 2) AS pendapatan
                 FROM perjalanan
                 WHERE wilayah_jemput = 'Manhattan'
                   AND waktu_jemput >= TIMESTAMP '{TANGGAL_AWAL}'
                   AND waktu_jemput <  TIMESTAMP '{TANGGAL_AKHIR}'
                 GROUP BY 1 ORDER BY 1"""
dasbor = ukur("kasus 2 - dasbor pendapatan per jam (DuckDB)", lambda: con_d.sql(SQL_DASBOR).df())
print(dasbor.head())

tujuan_tersering = ukur("kasus 3 - lima zona tujuan tersering (graph)",
                        lambda: sorted(G[zona_awal].items(), key=lambda x: -x[1]["jumlah"])[:5])
print("dari zona", zona_awal, "->", [(z, d["jumlah"]) for z, d in tujuan_tersering])

RADIUS_DERAJAT = 0.0045
pusat = points(-73.98, 40.75)
sekitar = ukur("kasus 4 - titik jemput dalam radius 500 meter (STRtree)",
               lambda: pohon.query(pusat.buffer(RADIUS_DERAJAT), predicate="contains"))
print("titik di dalam radius:", len(sekitar))

versi_awal = ukur("kasus 5 - membaca tabel pada versi 1 (time travel)",
                  lambda: DeltaTable(PATH_DELTA, version=1).to_pandas())
print(f"versi 1 -> {len(versi_awal):,} baris | total_bayar {round(versi_awal['total_bayar'].sum(), 2):,}")

[kasus 1 - detail perjalanan dari key-value] 0.0000 s
detail id 186265 -> {'id_perjalanan': 186265, 'waktu_jemput': '2023-01-10 11:50:57', 'zona_jemput': 170, 'zona_tujuan': 249, 'wilayah_jemput': 'Manhattan', 'jarak_mil': 1.9, 'total_bayar': 17.5, 'metode_bayar': 'Tunai'}
[kasus 2 - dasbor pendapatan per jam (DuckDB)] 0.0508 s
   jam  jumlah  pendapatan
0    0   15681   367889.65
1    1   11339   263777.65
2    2    7997   190204.40
3    3    4879   121093.90
4    4    3014    93080.35
[kasus 3 - lima zona tujuan tersering (graph)] 0.0003 s
dari zona 236 -> [(237, 18810), (236, 14110), (161, 7497), (239, 6877), (141, 6876)]
[kasus 4 - titik jemput dalam radius 500 meter (STRtree)] 0.0013 s
titik di dalam radius: 652
[kasus 5 - membaca tabel pada versi 1 (time travel)] 0.0388 s
versi 1 -> 150,000 baris | total_bayar 4,100,172.91


## P-4. Data Ganda pada Rancangan

Bagian rancangan yang terpaksa menyimpan data ganda adalah keempat penyimpanan turunan, yaitu dict kv, dokumen TinyDB, tabel arus pada graph, dan titik pada STRtree, karena semuanya menyalin sebagian isi trips_bersih ke bentuk yang berbeda. Salinan itu dijaga tetap sama dengan menjadikan trips_bersih sebagai satu-satunya sumber kebenaran dan membangun ulang seluruh turunan dari sumber yang sama setiap kali data bulan baru masuk, bukan menyunting tiap salinan satu per satu. Pembangunan ulang memakai kunci yang sama, yaitu id_perjalanan pada key-value dan dokumen serta pasangan zona_jemput dan zona_tujuan pada graph, sehingga hasilnya dapat disilangkan kembali seperti pada K-5 yang membandingkan lima arus terbesar versi graph dengan versi SQL. Untuk tabel tarif yang boleh dikoreksi, salinannya tidak dibuat sama sekali melainkan disimpan sebagai tabel Delta, karena koreksi di situ harus tercatat sebagai versi dan dapat dibatalkan, sedangkan penyimpanan turunan lain cukup dibangun ulang setelah koreksi diterima.

# Q. Latihan

## Latihan 1: Pencarian 2.000 Id pada Tabel Berindeks dan Key-value

Nilai *seed* diubah melalui random.seed(7) agar daftar id uji berbeda dengan K-2, lalu id_uji_besar dibentuk dari random.sample(range(1, len(data) + 1), 2_000). Fungsi cari_sql_besar() membuat kursor baru dengan con.cursor() lalu mengembalikan list hasil cur.execute() berparameter tanda tanya yang diikuti fetchone() untuk setiap id, sedangkan cari_kv_besar() mengembalikan list json.loads() atas kv pada kunci berpola perjalanan:<id>. Keduanya dijalankan di dalam ukur() dan durasinya diambil kembali dari catatan[-1]["detik"] lalu di-*assign* ke detik_sql dan detik_kv. Waktu rata-rata per pencarian dihitung dengan membagi tiap durasi dengan 2_000 lalu mengalikannya dengan 1e6, dan rasio detik_sql terhadap detik_kv dicetak sebagai pembanding.

Dugaan: waktu rata-rata per pencarian diperkirakan tetap pada kisaran yang sama dengan K-2 meskipun jumlah pencarian sepuluh kali lebih banyak, karena biaya pencarian pada indeks dan pada dict tidak bergantung pada banyaknya pencarian, dan key-value diperkirakan tetap beberapa kali lebih cepat.

In [ ]:
random.seed(7)
id_uji_besar = random.sample(range(1, len(data) + 1), 2_000)

def cari_sql_besar():
  cur = con.cursor()
  return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji_besar]

def cari_kv_besar():
  return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji_besar]

ukur("latihan 1 - 2.000 pencarian, SQL berindeks", cari_sql_besar)
detik_sql = catatan[-1]["detik"]
ukur("latihan 1 - 2.000 pencarian, key-value", cari_kv_besar)
detik_kv = catatan[-1]["detik"]

print(f"SQL berindeks : {detik_sql / 2_000 * 1e6:,.2f} mikrodetik per pencarian")
print(f"key-value     : {detik_kv / 2_000 * 1e6:,.2f} mikrodetik per pencarian")
print(f"key-value lebih cepat {detik_sql / detik_kv:,.1f} kali")

[latihan 1 - 2.000 pencarian, SQL berindeks] 0.0443 s
[latihan 1 - 2.000 pencarian, key-value] 0.0128 s
SQL berindeks : 22.15 mikrodetik per pencarian
key-value     : 6.40 mikrodetik per pencarian
key-value lebih cepat 3.5 kali


## Latihan 2: Dokumen dengan Zona Tujuan Tertentu dan Pembayaran Tunai

Konstanta ZONA_UJI ditetapkan bernilai 236, lalu dokumen dihitung dengan len() atas tabel.search() yang menggabungkan dua syarat Query melalui operator &, yaitu Q.tujuan.zona sama dengan ZONA_UJI dan Q.biaya.metode_bayar sama dengan Tunai. Angka yang sama dihitung lewat SQL pada tabel K-2 dengan con.execute() berisi SELECT COUNT(\*) yang disaring pada zona_tujuan dan metode_bayar, memakai parameter tanda tanya untuk ZONA_UJI, lalu diambil melalui fetchone() pada indeks ke-0. Kedua angka dicetak bersama ukuran populasinya, yaitu len(tabel) untuk dokumen dan len(data) untuk baris tabel, lalu kesamaannya diuji dengan operator ==.

Dugaan: kedua angka diperkirakan tidak sama karena dokumen TinyDB hanya dibangun dari sample.head(20_000) sedangkan tabel SQLite memuat seluruh 300.000 baris, sehingga jumlah dokumen diperkirakan sekitar seperlima belas jumlah baris SQL, bukan nol.

In [ ]:
ZONA_UJI = 236

jumlah_dokumen = len(tabel.search((Q.tujuan.zona == ZONA_UJI) & (Q.biaya.metode_bayar == "Tunai")))
jumlah_sql = con.execute(
    "SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan = ? AND metode_bayar = 'Tunai'",
    (ZONA_UJI,)).fetchone()[0]

print(f"dokumen TinyDB : {jumlah_dokumen:,} dari {len(tabel):,} dokumen")
print(f"baris SQLite   : {jumlah_sql:,} dari {len(data):,} baris")
print("sama?", jumlah_dokumen == jumlah_sql)

dokumen TinyDB : 155 dari 20,001 dokumen
baris SQLite   : 2,074 dari 300,000 baris
sama? False


## Jawaban Latihan 2

Kedua angka tidak sama, dan penyebabnya bukan perbedaan model data melainkan perbedaan populasi yang disimpan. Dokumen pada tabel dibangun dari sample.head(20_000) ditambah satu dokumen contoh yang disisipkan pada K-3, sedangkan tabel perjalanan pada SQLite memuat seluruh isi data, yaitu [isi dari keluaran] baris. Perbandingan yang adil menuntut kedua sisi dibangun dari populasi yang sama, misalnya dengan membatasi query SQL pada 20.000 id pertama. Pelajaran yang sama muncul pada K-5, yaitu dua sistem baru dapat disilangkan setelah dipastikan keduanya membaca baris yang sama.

## Latihan 3: SUM Satu Kolom dan SUM Beberapa Kolom pada DuckDB

Dua query disiapkan sebagai *string*, yaitu SQL_SATU berisi SELECT ROUND(SUM(total_bayar), 2) dan SQL_BANYAK berisi empat ROUND(SUM(...)) atas total_bayar, jarak_mil, zona_jemput, dan zona_tujuan, keduanya atas view perjalanan. Masing-masing dijalankan melalui con_d.sql().fetchall() yang dibungkus ukur(), lalu durasinya diambil dari catatan[-1]["detik"] dan di-*assign* ke detik_satu dan detik_banyak. Hasil kedua query dicetak, kemudian rasio detik_banyak terhadap detik_satu dicetak dengan satu angka desimal sebagai ukuran tambahan biaya karena jumlah kolom yang dibaca bertambah.

Dugaan: SUM atas empat kolom diperkirakan dua sampai empat kali lebih lama daripada SUM atas satu kolom, karena *column pruning* membuat biaya query hampir sebanding dengan jumlah kolom yang benar-benar dibaca, bukan dengan jumlah kolom pada berkas.

In [ ]:
SQL_SATU = "SELECT ROUND(SUM(total_bayar), 2) FROM perjalanan"
SQL_BANYAK = """SELECT ROUND(SUM(total_bayar), 2), ROUND(SUM(jarak_mil), 2),
                       ROUND(SUM(zona_jemput), 2), ROUND(SUM(zona_tujuan), 2)
                FROM perjalanan"""

hasil_satu = ukur("latihan 3 - SUM satu kolom", lambda: con_d.sql(SQL_SATU).fetchall())
detik_satu = catatan[-1]["detik"]
hasil_banyak = ukur("latihan 3 - SUM empat kolom", lambda: con_d.sql(SQL_BANYAK).fetchall())
detik_banyak = catatan[-1]["detik"]

print("satu kolom  :", hasil_satu)
print("empat kolom :", hasil_banyak)
print(f"empat kolom memakan {detik_banyak / detik_satu:,.1f} kali waktu satu kolom")

[latihan 3 - SUM satu kolom] 0.0743 s
[latihan 3 - SUM empat kolom] 0.1712 s
satu kolom  : [(81621442.46,)]
empat kolom : [(81621442.46, 10231889.61, 497452755, 491330400)]
empat kolom memakan 2.3 kali waktu satu kolom


## Latihan 4: Lima Zona dengan Zona Tujuan Berbeda Terbanyak

Pada sisi graph, G.out_degree() mengembalikan pasangan zona dan jumlah edge keluar yang diurutkan melalui sorted() dengan key berupa *lambda* bernilai negatif jumlah lalu nomor zona sebagai pengurut tambahan, kemudian dipotong lima teratas dengan *slicing*. Pada sisi SQL, con_d.sql() menjalankan SELECT asal dengan COUNT(DISTINCT tujuan) atas tabel arus yang sudah didaftarkan pada K-5, dikelompokkan GROUP BY asal, diurutkan ORDER BY jumlah_tujuan DESC lalu asal, dibatasi LIMIT 5, dan diubah ke DataFrame dengan df(). Hasil SQL diubah menjadi list pasangan melalui itertuples(index=False, name=None), nilainya di-*cast* dengan int() agar sebanding dengan tipe pada graph, lalu kedua list dibandingkan dengan operator ==.

Dugaan: kedua sisi diperkirakan menghasilkan lima zona yang sama persis karena out-degree pada graph dibangun dari tabel arus yang sama, dan pengurut tambahan berupa nomor zona diperkirakan menjadi penentu urutan ketika ada zona dengan jumlah tujuan yang sama.

In [ ]:
top_graf = sorted(G.out_degree(), key=lambda kv: (-kv[1], kv[0]))[:5]
print("graph:", top_graf)

top_sql = con_d.sql("""SELECT asal, COUNT(DISTINCT tujuan) AS jumlah_tujuan
                       FROM arus GROUP BY asal
                       ORDER BY jumlah_tujuan DESC, asal LIMIT 5""").df()
pasangan_sql = [(int(a), int(b)) for a, b in top_sql.itertuples(index=False, name=None)]
print("SQL  :", pasangan_sql)
print("sama?", top_graf == pasangan_sql)

graph: [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
SQL  : [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
sama? True


## Latihan 5: Pengaruh Ukuran Kotak terhadap Keuntungan Indeks

Didefinisikan fungsi bandingkan_kotak(sisi) yang menghitung separuh sisi lalu membentuk kotak_uji melalui *list comprehension* atas zip(cx, cy) dengan box() pada batas x dan y yang digeser separuh ke kiri, kanan, bawah, dan atas. Di dalamnya didefinisikan cara_numpy_uji() yang menjalankan *for loop* atas zip(cx, cy), menyusun *boolean mask* m dari empat perbandingan atas bujur dan lintang yang digabung operator &, lalu menambahkan int(m.sum()) ke list hasil. Kedua cara dijalankan di dalam ukur() dengan label yang memuat nilai sisi, durasinya diambil dari catatan[-1]["detik"], dan pencarian dengan indeks memakai *list comprehension* atas kotak_uji yang memanggil pohon.query() dengan predicate="contains". Fungsi mencetak jumlah titik yang ditemukan melalui sum(), kesamaan kedua hasil dengan operator ==, dan rasio detik_np terhadap detik_idx, lalu dipanggil berturut-turut untuk sisi 0,02, 0,01, dan 0,001 derajat di dalam *for loop*.

Dugaan: keuntungan STRtree terhadap NumPy diperkirakan membesar ketika kotak mengecil, karena NumPy tetap membandingkan seluruh 200.000 titik berapa pun ukuran kotaknya, sedangkan STRtree melewati kelompok kotak yang tidak bersinggungan sehingga jumlah titik yang benar-benar diperiksa ikut mengecil.

In [ ]:
def bandingkan_kotak(sisi):
  separuh = sisi / 2
  kotak_uji = [box(x - separuh, y - separuh, x + separuh, y + separuh) for x, y in zip(cx, cy)]

  def cara_numpy_uji():
    hasil = []
    for x, y in zip(cx, cy):
      m = (bujur >= x - separuh) & (bujur <= x + separuh) & (lintang >= y - separuh) & (lintang <= y + separuh)
      hasil.append(int(m.sum()))
    return hasil

  h_np_uji = ukur(f"latihan 5 - NumPy, sisi {sisi}", cara_numpy_uji)
  detik_np = catatan[-1]["detik"]
  h_idx_uji = ukur(f"latihan 5 - STRtree, sisi {sisi}",
                   lambda: [len(pohon.query(k, predicate="contains")) for k in kotak_uji])
  detik_idx = catatan[-1]["detik"]

  print(f"sisi {sisi} derajat: {sum(h_idx_uji):,} titik | hasil sama? {h_np_uji == h_idx_uji}"
        f" | STRtree {detik_np / detik_idx:,.1f} kali NumPy")

for sisi in (0.02, 0.01, 0.001):
  bandingkan_kotak(sisi)

[latihan 5 - NumPy, sisi 0.02] 0.5592 s
[latihan 5 - STRtree, sisi 0.02] 0.8440 s
sisi 0.02 derajat: 1,666,631 titik | hasil sama? True | STRtree 0.7 kali NumPy
[latihan 5 - NumPy, sisi 0.01] 0.5617 s
[latihan 5 - STRtree, sisi 0.01] 0.2424 s
sisi 0.01 derajat: 417,545 titik | hasil sama? True | STRtree 2.3 kali NumPy
[latihan 5 - NumPy, sisi 0.001] 0.5568 s
[latihan 5 - STRtree, sisi 0.001] 0.0136 s
sisi 0.001 derajat: 4,149 titik | hasil sama? True | STRtree 40.9 kali NumPy


## Jawaban Latihan 5

Perbandingan STRtree terhadap NumPy membesar ketika sisi kotak mengecil, dari [isi dari keluaran] kali pada sisi 0,02 derajat menjadi [isi dari keluaran] kali pada sisi 0,001 derajat. Sebabnya, biaya cara_numpy_uji() hampir tetap karena keempat perbandingan selalu dikenakan pada seluruh 200.000 nilai bujur dan lintang berapa pun ukuran kotaknya, sedangkan pohon.query() memakai kotak berjenjang STRtree untuk melewati kelompok titik yang tidak bersinggungan, sehingga jumlah titik yang benar-benar diperiksa menyusut bersama luas kotak. Kotak yang lebih kecil juga membuat jumlah titik yang ditemukan mengecil, sehingga pekerjaan yang tersisa setelah penyaringan ikut berkurang.

## Latihan 6: Restore Tabel Delta ke Versi Lama

Objek dt dibentuk dari DeltaTable(PATH_DELTA), lalu versi dan jumlah barisnya dicetak melalui dt.version() dan len(dt.to_pandas()) sebagai keadaan sebelum restore. Perintah dt.restore(1) mengembalikan isi tabel ke keadaan versi 1, objek dt dibentuk ulang agar membaca catatan terbaru, lalu versi dan jumlah barisnya dicetak kembali. Tiga operasi terakhir pada riwayat ditampilkan melalui *list comprehension* atas dt.history() yang memanggil h.get("operation") dan dipotong dengan *slicing* tiga teratas.

Dugaan: jumlah baris diperkirakan kembali ke jumlah pada versi 1, sedangkan nomor versi diperkirakan justru bertambah satu karena restore dicatat sebagai operasi baru, sehingga riwayat diperkirakan diawali RESTORE dan tetap memuat OPTIMIZE serta WRITE di bawahnya.

In [ ]:
dt = DeltaTable(PATH_DELTA)
print("versi sebelum restore:", dt.version(), "| baris:", f"{len(dt.to_pandas()):,}")

dt.restore(1)

dt = DeltaTable(PATH_DELTA)
print("versi setelah restore:", dt.version(), "| baris:", f"{len(dt.to_pandas()):,}")
print([h.get("operation") for h in dt.history()][:3])

versi sebelum restore: 23 | baris: 160,000
versi setelah restore: 24 | baris: 150,000
['RESTORE', 'OPTIMIZE', 'WRITE']


## Jawaban Latihan 6

Time travel hanya membaca, sedangkan restore menulis. DeltaTable(PATH_DELTA, version=1) pada K-7 menghasilkan DataFrame berisi keadaan lama tanpa mengubah tabel, sehingga versi terbarunya tetap. Perintah dt.restore(1) mengubah tabel aktif menjadi isi versi 1 dan perubahannya sendiri dicatat sebagai versi baru, yaitu nomor versi naik dari [isi dari keluaran] menjadi [isi dari keluaran] dengan operasi RESTORE pada riwayat. Restore dicatat sebagai versi baru, bukan menghapus riwayat, karena _delta_log bersifat menambah catatan sebagaimana dijelaskan bagian M.6 modul, sehingga keadaan sebelum restore pun masih dapat dibaca kembali dan kekeliruan pada restore itu sendiri masih dapat dibatalkan.

# R. Tugas Praktikum

## Tugas 1. Rekap Benchmark dan Tabel Pilihan Penyimpanan

Seluruh pengukuran yang terkumpul sampai sel ini, termasuk tambahan dari bagian O, P, dan Q, diubah menjadi DataFrame rekap melalui pd.DataFrame(catatan) lalu disimpan ulang ke DIR_SIMPAN sebagai benchmark_nosql.csv dengan to_csv() berargumen index=False, dan jumlah barisnya dicetak dengan len(). Series waktu dibentuk ulang melalui drop_duplicates("langkah", keep="last") dan set_index("langkah") agar memuat label terbaru, lalu fungsi susun_rekap_pilihan() yang didefinisikan pada K-9 dipanggil kembali dengan Series itu sehingga tabel rekap terisi angka yang sama dengan isi benchmark terakhir. Hasilnya disimpan sebagai rekap_pilihan_penyimpanan.csv dengan index=False lalu ditampilkan sebagai keluaran sel.

Dugaan: jumlah baris benchmark_nosql.csv diperkirakan hampir dua kali jumlah baris pada K-9 karena bagian O, P, dan Q menambahkan pengukurannya sendiri, sedangkan empat baris tabel rekap yang berangka diperkirakan tidak berubah karena label K-2 sampai K-6 tidak diukur ulang.

In [ ]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print("baris pengukuran yang tersimpan:", len(rekap))

waktu = (pd.DataFrame(catatan)
         .drop_duplicates("langkah", keep="last")
         .set_index("langkah")["detik"])

rekap_pilihan = susun_rekap_pilihan(waktu)
rekap_pilihan.to_csv(f"{DIR_SIMPAN}/rekap_pilihan_penyimpanan.csv", index=False)
rekap_pilihan

baris pengukuran yang tersimpan: 34


,kebutuhan,cara_unggul,cara_lemah,angka
0,Ambil satu data lewat id,Key-value; tabel dengan indeks,Tabel tanpa indeks,"6,277 kali"
1,Menghitung atau merangkum banyak baris,Per kolom (DuckDB pada Parquet),Key-value (buka semua nilai),key-value 7.3 kali SQL; SQLite 27.8 kali DuckDB
2,Data dengan field opsional atau berubah-ubah,Document,"Tabel (kolom NULL, migrasi)",kualitatif: field cuaca masuk tanpa ALTER TABLE
3,Keterhubungan dengan jumlah langkah bebas,Graph,SQL (satu JOIN per langkah),1.1 kali pada dua langkah
4,Mencari lokasi dalam wilayah,Spasial dengan indeks,Memeriksa semua titik,2.3 kali terhadap NumPy
5,Perubahan yang harus aman dan bisa dibatalkan,Tabel Delta,Folder Parquet biasa,"kualitatif: time travel, penolakan skema, dan ..."


## Tugas 2. Data Lain dengan Tiga Cara Menyimpan

Data lain yang dipilih adalah zona, yaitu ringkasan per zona jemput. Ringkasan dibentuk dengan con_d.sql() berisi SELECT zona_jemput, MIN(wilayah_jemput), COUNT(\*), dan ROUND(AVG(total_bayar), 2) yang dikelompokkan GROUP BY zona_jemput serta diurutkan ORDER BY zona, lalu diubah ke DataFrame ringkas_zona melalui df(). Cara pertama menyimpannya sebagai tabel dengan to_sql("zona", con, index=False, if_exists="replace") lalu memeriksanya dengan con.execute() yang mengambil tiga zona tersibuk melalui ORDER BY jumlah_jemput DESC LIMIT 3. Cara kedua menyimpannya sebagai dokumen pada db.table("zona") yang dikosongkan lebih dahulu dengan truncate(), kemudian *for loop* atas ringkas_zona.itertuples(index=False) menyusun daftar tujuan_teratas dari G[zona] bila zona ada di dalam G melalui sorted() dengan key berupa *lambda*, dan memasukkan dokumen bersarang melalui insert(). Cara ketiga menyimpannya sebagai key-value, yaitu dict kv_zona yang kuncinya berpola zona:<id> dan nilainya teks JSON dari json.dumps(). Jumlah isi ketiga bentuk dicetak, dan satu contoh dokumen ditampilkan melalui json.dumps() berargumen indent=2.

Dugaan: ketiga bentuk diperkirakan menyimpan jumlah zona yang sama, sedangkan perbedaannya diperkirakan baru terlihat pada bentuk pertanyaannya, yaitu tabel unggul untuk pengurutan dan agregasi, dokumen unggul untuk membaca satu zona beserta tujuan teratasnya sekaligus, dan key-value unggul untuk pengambilan satu zona lewat kuncinya.

In [ ]:
ringkas_zona = con_d.sql("""SELECT zona_jemput AS zona, MIN(wilayah_jemput) AS wilayah,
                                   COUNT(*) AS jumlah_jemput, ROUND(AVG(total_bayar), 2) AS rata_bayar
                            FROM perjalanan GROUP BY zona_jemput ORDER BY zona""").df()

ringkas_zona.to_sql("zona", con, index=False, if_exists="replace")
tersibuk = con.execute("SELECT zona, wilayah, jumlah_jemput FROM zona ORDER BY jumlah_jemput DESC LIMIT 3").fetchall()

tabel_zona = db.table("zona")
tabel_zona.truncate()
for r in ringkas_zona.itertuples(index=False):
  zona = int(r.zona)
  tujuan_teratas = sorted(G[zona].items(), key=lambda x: -x[1]["jumlah"])[:3] if zona in G else []
  tabel_zona.insert({"zona": zona, "wilayah": r.wilayah,
                     "jumlah_jemput": int(r.jumlah_jemput), "rata_bayar": float(r.rata_bayar),
                     "tujuan_teratas": [{"zona": int(z), "jumlah": int(d["jumlah"])} for z, d in tujuan_teratas]})

kv_zona = {f"zona:{int(r.zona)}": json.dumps({"wilayah": r.wilayah,
                                              "jumlah_jemput": int(r.jumlah_jemput),
                                              "rata_bayar": float(r.rata_bayar)})
           for r in ringkas_zona.itertuples(index=False)}

print("tabel   :", len(ringkas_zona), "baris | tiga zona tersibuk:", tersibuk)
print("document:", len(tabel_zona), "dokumen")
print("key-value:", len(kv_zona), "kunci | contoh:", kv_zona["zona:" + str(int(ringkas_zona['zona'].iloc[0]))])
print(json.dumps(tabel_zona.all()[0], indent=2))

tabel   : 254 baris | tiga zona tersibuk: [(132, 'Queens', 152840), (237, 'Manhattan', 145797), (236, 'Manhattan', 136276)]
document: 254 dokumen
key-value: 254 kunci | contoh: {"wilayah": "EWR", "jumlah_jemput": 28, "rata_bayar": 106.41}
{
  "zona": 1,
  "wilayah": "EWR",
  "jumlah_jemput": 28,
  "rata_bayar": 106.41,
  "tujuan_teratas": [
    {
      "zona": 1,
      "jumlah": 12
    },
    {
      "zona": 265,
      "jumlah": 6
    },
    {
      "zona": 6,
      "jumlah": 2
    }
  ]
}


## Jawaban Tugas 2

Pada bentuk tabel, pertanyaan yang mudah adalah mengurutkan seluruh zona menurut jumlah_jemput atau menghitung rata-rata rata_bayar per wilayah, karena cukup satu SELECT dengan ORDER BY atau GROUP BY. Pertanyaan yang canggung adalah menampilkan satu zona beserta tiga tujuan teratasnya, karena daftar itu tidak muat di dalam satu baris dan menuntut tabel kedua beserta JOIN.

Pada bentuk document, pertanyaan yang mudah adalah membuka satu zona lengkap dengan tujuan_teratas-nya dalam satu kali baca, karena daftar tujuan sudah disatukan di dalam dokumen (*embedding*). Pertanyaan yang canggung adalah menghitung rata-rata rata_bayar seluruh zona, karena setiap dokumen harus dibuka satu per satu seperti pada hitung_kv() di K-2.

Pada bentuk key-value, pertanyaan yang mudah adalah mengambil ringkasan satu zona melalui kunci zona:<id> tanpa pencarian. Pertanyaan yang canggung adalah menemukan zona dengan jumlah_jemput terbesar, karena kunci tidak menyimpan apa pun tentang isinya sehingga seluruh nilai harus dibuka lebih dahulu.

## Tugas 3. Simulasi Delta Lake Dua Hari Kerja

Tabel baru disiapkan pada PATH_HARIAN di disk runtime, sama seperti tabel Delta pada K-7, dan sisa percobaan sebelumnya dihapus dengan shutil.rmtree() apabila os.path.exists() bernilai True, sehingga sel aman dijalankan ulang. Data hari pertama diambil dari data_delta.iloc[:60_000] dan ditulis dengan write_deltalake() bermode overwrite, lalu versi dan jumlah barisnya dicetak dari DeltaTable(PATH_HARIAN).version() dan len(to_pandas()). Hari kedua menambahkan data_delta.iloc[60_000:90_000] dengan mode append, menghitung jumlah baris yang akan dikoreksi dari gabungan data_delta.iloc[:90_000] melalui dua *boolean mask* atas wilayah_jemput dan metode_bayar yang digabung operator & lalu dijumlahkan dengan sum() dan di-*cast* int(), kemudian menjalankan dt_harian.update() berpredikat wilayah_jemput Queens dan metode_bayar Tunai dengan updates yang menambah total_bayar sebesar 0.5. Keadaan akhir hari pertama dibaca kembali melalui DeltaTable(PATH_HARIAN, version=0).to_pandas(), jumlah baris dan jumlah total_bayar-nya dicetak, lalu isi _delta_log ditampilkan melalui sorted(os.listdir()).

Keputusan pada sel ini, koreksi ditetapkan pada wilayah Queens dengan penambahan 0,5 agar berbeda dengan koreksi Bronx pada K-7, sehingga selisih yang muncul dapat dipastikan berasal dari simulasi ini, dengan konsekuensi angkanya tidak dapat dibandingkan langsung dengan n_koreksi pada K-7.

Dugaan: pembacaan versi 0 diperkirakan tetap menghasilkan 60.000 baris dengan jumlah total_bayar sebelum koreksi, dan _delta_log diperkirakan memuat tiga berkas JSON, yaitu satu untuk penulisan hari pertama, satu untuk penambahan hari kedua, dan satu untuk update.

In [ ]:
PATH_HARIAN = "/content/trips_harian"
if os.path.exists(PATH_HARIAN):
  shutil.rmtree(PATH_HARIAN)

hari_pertama = data_delta.iloc[:60_000]
write_deltalake(PATH_HARIAN, hari_pertama, mode="overwrite")
print("akhir hari pertama -> versi:", DeltaTable(PATH_HARIAN).version(),
      "| baris:", f"{len(DeltaTable(PATH_HARIAN).to_pandas()):,}")

write_deltalake(PATH_HARIAN, data_delta.iloc[60_000:90_000], mode="append")
dua_hari = data_delta.iloc[:90_000]
n_koreksi_harian = int(((dua_hari["wilayah_jemput"] == "Queens") & (dua_hari["metode_bayar"] == "Tunai")).sum())
dt_harian = DeltaTable(PATH_HARIAN)
dt_harian.update(predicate="wilayah_jemput = 'Queens' AND metode_bayar = 'Tunai'",
                 updates={"total_bayar": "total_bayar + 0.5"})
print("akhir hari kedua   -> versi:", DeltaTable(PATH_HARIAN).version(),
      "| baris:", f"{len(DeltaTable(PATH_HARIAN).to_pandas()):,}", "| dikoreksi:", f"{n_koreksi_harian:,}")

akhir_hari_pertama = DeltaTable(PATH_HARIAN, version=0).to_pandas()
print(f"dibaca ulang keadaan akhir hari pertama -> {len(akhir_hari_pertama):,} baris"
      f" | total_bayar {round(akhir_hari_pertama['total_bayar'].sum(), 2):,}")
print("isi _delta_log:", sorted(os.listdir(f"{PATH_HARIAN}/_delta_log")))

akhir hari pertama -> versi: 0 | baris: 60,000
akhir hari kedua   -> versi: 2 | baris: 90,000 | dikoreksi: 1,851
dibaca ulang keadaan akhir hari pertama -> 60,000 baris | total_bayar 1,644,649.69
isi _delta_log: ['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']


## Tugas 4. Laporan Rancangan dan Pemeriksaan Akhir

Tugas keempat tidak memerlukan kode baru karena berupa penyusunan laporan dan pemeriksaan notebook. Laporan rancangan penyimpanan untuk studi kasus Bagian P disusun sepanjang dua sampai tiga halaman dengan tabel yang kolomnya mengikuti modul, yaitu cara menyimpan yang dipilih, alasan yang mengutip angka dari benchmark_nosql.csv, dan kelemahan yang harus diwaspadai, ditambah satu diagram aliran data dan satu paragraf tentang data ganda. Isi tabel dan diagram itu sudah dihasilkan sel P-1 sampai P-4 sebagai rancangan_penyimpanan.csv dan diagram_rancangan_penyimpanan.png.

Sebelum dikumpulkan, notebook dijalankan ulang melalui Runtime lalu Restart session and run all pada *runtime* bersih agar seluruh keluaran berasal dari satu kali eksekusi berurutan. Berkas yang dikumpulkan adalah notebook .ipynb beserta keluarannya, benchmark_nosql.csv, arsip trips_delta.zip, dan laporan praktikum dalam bentuk PDF.

## Tugas 4. Mengarsipkan Tabel Delta untuk Pengumpulan

Folder tabel Delta yang berada di disk runtime diarsipkan ke Google Drive melalui shutil.make_archive() dengan base_name berupa trips_delta di dalam DIR_SIMPAN, format zip, dan root_dir berupa PATH_DELTA, lalu jalur berkas hasilnya di-*assign* ke ARSIP. Langkah inilah yang memindahkan tabel Delta ke penyimpanan permanen, karena tabelnya sendiri tidak dapat ditulis langsung di Drive. Ukuran arsip dicetak dalam MB dari os.path.getsize(ARSIP) yang dibagi 1024 pangkat dua dan dibulatkan round() dua desimal, kemudian seluruh isi DIR_SIMPAN didaftar melalui sorted(os.listdir(DIR_SIMPAN)) sebagai pemeriksaan terakhir bahwa berkas yang dikumpulkan sudah lengkap.

Dugaan: ukuran arsip diperkirakan hampir sama dengan ukuran folder aslinya karena berkas Parquet di dalamnya sudah terkompresi, sehingga penyusutan hanya berasal dari berkas JSON pada _delta_log, dan daftar isi DIR_SIMPAN diperkirakan memuat tiga berkas CSV, satu berkas PNG, dan satu arsip zip, tanpa folder tabel Delta karena keduanya berada di disk runtime.

In [ ]:
ARSIP = shutil.make_archive(f"{DIR_SIMPAN}/trips_delta", "zip", PATH_DELTA)
print("arsip dibuat:", ARSIP, "|", round(os.path.getsize(ARSIP) / 1024 ** 2, 2), "MB")
print("isi DIR_SIMPAN:", sorted(os.listdir(DIR_SIMPAN)))

arsip dibuat: /content/drive/MyDrive/BigData/Praktikum4/trips_delta.zip | 5.72 MB
isi DIR_SIMPAN: ['benchmark_nosql.csv', 'diagram_rancangan_penyimpanan.png', 'rancangan_penyimpanan.csv', 'rekap_pilihan_penyimpanan.csv', 'trips_delta.zip']


# S. Pertanyaan Evaluasi

## S-1. Arti Not Only SQL dan Kapan Tabel Relasional Tetap Lebih Baik

NoSQL dibaca Not Only SQL karena bukan pengganti SQL, melainkan tambahan pilihan di samping tabel relasional, sebagaimana dinyatakan bagian G.1 modul. Sistem sungguhan biasanya memakai beberapa jenis penyimpanan sekaligus sesuai pekerjaan yang paling cocok bagi masing-masing, yaitu pola yang disebut *polyglot persistence*, dan praktikum ini memperlihatkannya ketika satu data perjalanan yang sama disimpan dalam lima bentuk berbeda.

Situasi pertama yang tetap menuntut tabel relasional adalah data yang bentuknya tetap dan hubungannya banyak ke banyak, misalnya tarif, zona, dan metode pembayaran yang saling dirujuk, karena JOIN dan kunci asing mengerjakannya tanpa menduplikasi data, sedangkan model document harus memilih antara menyatukan atau merujuk sebagaimana dibahas bagian M.3. Situasi kedua adalah transaksi yang harus benar seketika dan melibatkan lebih dari satu baris, misalnya pemotongan saldo penumpang bersama pencatatan pembayaran pengemudi, karena jaminan ACID pada tabel relasional memastikan keduanya berhasil bersama atau gagal bersama, sedangkan sifat BASE pada banyak sistem NoSQL hanya menjanjikan keseragaman data pada akhirnya.

## S-2. BASE, ACID, dan Teorema CAP pada Aplikasi Taksi

ACID menuntut setiap transaksi bersifat atomik, konsisten, terisolasi, dan tahan gagal, sehingga pembaca tidak pernah melihat keadaan setengah jadi. BASE menempuh jalan sebaliknya, yaitu mengutamakan ketersediaan dan kecepatan dengan menerima bahwa salinan data pada beberapa mesin baru seragam setelah beberapa saat, yang disebut *eventually consistent*.

Teorema CAP menyatakan bahwa ketika jaringan antarmesin terputus, sistem harus memilih antara *consistency*, yaitu semua pembaca melihat data terbaru yang sama, atau *availability*, yaitu setiap permintaan tetap dijawab meskipun datanya mungkin tertinggal. Pada aplikasi taksi, data pembayaran dan saldo penumpang sebaiknya memilih consistency, karena tarif yang terbaca berbeda antarmesin dapat menimbulkan penagihan ganda atau selisih pembukuan. Sebaliknya, daftar perjalanan terbaru pada riwayat penumpang boleh memilih availability, karena perjalanan yang baru selesai dan belum muncul selama beberapa detik tidak menimbulkan kerugian, sedangkan halaman yang gagal dimuat justru mengganggu. Tabel Delta pada K-7 berada di sisi consistency, yaitu satu komit ditulis sekaligus sehingga pembaca melihat tabel sebelum atau sesudah perubahan, tidak pernah di tengahnya.

## S-3. Mengapa Key-value Cepat Mencari tetapi Lambat Menghitung

Pencarian satu data pada key-value sangat cepat karena kamus Python menghitung letak nilai langsung dari kuncinya, sehingga jumlah data yang tersimpan hampir tidak memengaruhi waktu pengambilan. Hal itu terlihat pada K-2, yaitu 200 pencarian pada kv selesai dalam [isi dari keluaran O-1] mikrodetik per pencarian, sedangkan tabel tanpa indeks memerlukan [isi dari keluaran O-1] mikrodetik karena harus memeriksa baris demi baris sampai id yang diminta ditemukan.

Perhitungan menjadi lambat karena penyimpan key-value tidak mengetahui apa pun tentang isi nilainya, sebagaimana dijelaskan bagian M.2 modul. Nilai itu hanyalah sepotong teks, sehingga hitung_kv() harus membuka seluruh nilai dengan json.loads(), mengambil field wilayah_jemput dan total_bayar, lalu menjumlahkannya sendiri, dan hasilnya [isi dari keluaran O-2] kali lebih lambat daripada SQL GROUP BY yang mengetahui total_bayar sebagai kolom angka. Yang dikorbankan key-value demi kesederhanaannya adalah pengetahuan tentang isi data, yaitu skema, tipe kolom, indeks sekunder, dan kemampuan agregasi, dan sebagai gantinya ia memperoleh kunci yang mudah dibagi ke banyak mesin.

## S-4. Menyatukan dan Merujuk pada Model Document

Menyatukan data (*embedding*) berarti menaruh data terkait di dalam satu dokumen, sebagaimana dilakukan ke_dokumen() pada K-3 yang menyimpan zona dan wilayah penjemputan di dalam objek jemput pada dokumen perjalanan. Merujuk (*referencing*) berarti dokumen hanya menyimpan pengenalnya, misalnya nomor zona saja, sedangkan keterangan zona disimpan pada koleksi terpisah dan diambil melalui pencarian kedua.

Menyatukan lebih tepat ketika data terkait selalu dibaca bersama induknya dan jarang berubah, misalnya wilayah penjemputan pada dokumen perjalanan, karena satu kali baca sudah memberi semua yang dibutuhkan halaman detail perjalanan. Merujuk lebih tepat ketika data terkait sering berubah atau dipakai banyak dokumen, misalnya tarif dasar per zona, karena perubahan cukup dilakukan pada satu tempat, sedangkan pada bentuk yang disatukan seluruh dokumen yang memuatnya harus diperbarui. Harga merujuk adalah langkah pencarian tambahan, sedangkan harga menyatukan adalah data ganda yang harus dijaga tetap sama, persoalan yang juga muncul pada rancangan Bagian P.

## S-5. Keunggulan Graph yang Tidak Bergantung pada Kecepatan

Pada K-5 graph dan SQL *self-join* menjawab pertanyaan dua langkah dengan waktu yang hampir sama, sehingga alasan memilih graph tidak terletak pada kecepatan. Keunggulannya terletak pada bentuk pertanyaannya, yaitu jumlah langkah menjadi satu angka pada argumen cutoff sehingga pertanyaan satu sampai empat langkah dijawab oleh kode yang sama, sedangkan pada SQL setiap langkah tambahan menuntut satu JOIN tambahan sehingga pertanyaan tiga langkah harus ditulis sebagai query yang berbeda.

Keunggulan kedua adalah pertanyaan yang tidak diketahui panjangnya sebelumnya, misalnya jalur terpendek antara dua zona atau komponen yang saling terhubung, dapat dijawab dengan satu pemanggilan seperti nx.single_source_shortest_path_length(), sedangkan pada SQL pertanyaan itu menuntut query rekursif. Keunggulan ketiga adalah bobot dan atribut hubungan tersimpan pada edge, sehingga penyaringan MIN sebesar 50 cukup dibaca dari atribut jumlah tanpa mengubah bentuk penyimpanan. Dengan demikian graph dipilih karena pertanyaannya menjadi mudah dinyatakan dan mudah diubah, bukan karena hasilnya lebih cepat pada 263 zona.

## S-6. Cara Kerja Indeks Spasial dan Klaim Percepatan yang Menyesatkan

Sebelum memeriksa lokasi secara tepat, indeks spasial mengelompokkan titik yang berdekatan ke dalam kotak pembatas berjenjang, sebagaimana dijelaskan bagian G.5 modul. Saat pencarian dijalankan, kelompok yang kotaknya tidak bersinggungan dengan wilayah yang dicari dilewati seluruhnya tanpa satu titik pun diperiksa, dan hanya titik di dalam kelompok yang bersinggungan yang diuji dengan pemeriksaan geometri yang sebenarnya melalui predicate="contains". Cara kerja ini sejajar dengan *partition pruning* pada Praktikum 3, hanya saja pemisahannya berdasarkan letak di peta, bukan nilai kolom.

Klaim indeks seribu kali lebih cepat menyesatkan karena angka sebesar itu diperoleh dengan membandingkan indeks terhadap cara terburuk, yaitu loop Python yang memeriksa 200.000 titik satu per satu di dalam penerjemah. Terhadap pembanding yang wajar, yaitu NumPy yang membandingkan seluruh *array* sekaligus, keunggulannya menyusut menjadi [isi dari keluaran O-6] kali, dan menyusut lagi apabila waktu membangun indeks ikut dihitung. Karena itu setiap klaim percepatan harus menyebut pembandingnya, dan pembanding yang jujur adalah cara terbaik yang masuk akal, bukan cara terburuk yang mudah dikalahkan.

## S-7. Isi _delta_log, Sifat Atomik, dan Bahaya VACUUM Retention Nol

Folder _delta_log berisi catatan transaksi, yaitu satu berkas JSON untuk setiap versi tabel yang memuat daftar berkas Parquet yang ditambahkan dan yang dikeluarkan pada komit itu, sebagaimana dijelaskan bagian M.6 modul. Isi tabel pada suatu versi adalah hasil membaca catatan itu dari awal sampai versi yang diminta, sehingga folder log inilah sumber kebenarannya, sedangkan berkas Parquet hanyalah isinya.

Penulisan bersifat atomik karena satu komit adalah satu berkas JSON yang ditulis sekaligus. Pembaca yang datang sebelum berkas itu ada akan melihat tabel versi lama secara utuh, dan yang datang sesudahnya melihat versi baru secara utuh, tidak pernah melihat keadaan setengah jadi. Hal itu terbukti pada K-7, yaitu penambahan data berkolom asing ditolak dengan SchemaMismatchError dan nomor versi tidak berubah sama sekali. Time travel mungkin karena update dan compaction tidak menghapus berkas lama melainkan hanya mencatat bahwa berkas itu tidak lagi termasuk tabel, sehingga membaca versi sebelumnya cukup dengan membaca catatan sampai versi tersebut.

VACUUM dengan retention nol jam berbahaya karena perintah itu menghapus secara fisik seluruh berkas yang tidak lagi termasuk versi terbaru, termasuk berkas yang sedang dibaca proses lain dan berkas yang dibutuhkan versi lama. Akibatnya pekerjaan yang sedang berjalan dapat gagal dan time travel ke versi sebelumnya terputus, sehingga K-8 menjalankannya dengan dry_run=True yang hanya melaporkan calon berkas tanpa menghapus apa pun, dan nilai bawaan sekitar tujuh hari memang ada untuk melindungi hal itu.